In [ ]:

"""
SAM/SPIDER sezgiseli + gercek veri entegrasyonu
- Google Routes API tabanli statik mesafe/sure matrisi
- Split deliveries
- Heterogeneous fleet
- Region compatibility
- Gerektiginde ayni fiziksel aracin tekrar tur yapabilmesi
- Tekrar tur sadece fiziksel filodaki ilk turlar yetmezse tercih edilir
- Trafik bagimli sureler devre disidir
- Haritali cikti ve detayli Excel ciktilari korunur
"""

import os
import math
import random
import time
import html
from collections import defaultdict
from datetime import datetime, date, time as dtime, timedelta, timezone

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
try:
    from urllib3.util.retry import Retry
except Exception:
    Retry = None

try:
    import folium
except ImportError:
    folium = None


# =========================
# USER PARAMETERS
# =========================
BASE_DIR = os.path.expanduser("~/Desktop/VRP2")
FILES = {
    "locations": os.path.join(BASE_DIR, "2_lokasyon_bilgileri.xlsx"),
    "delivery": os.path.join(BASE_DIR, "3_delivery_bilgileri.xlsx"),
    "pickup": os.path.join(BASE_DIR, "4_pickup_bilgileri.xlsx"),
    "vehicles": os.path.join(BASE_DIR, "5_arac_bilgileri.xlsx"),
    "restrictions": os.path.join(BASE_DIR, "6_kisitli_bolgeler.xlsx"),
    "product_volume": os.path.join(BASE_DIR, "7_urun_hacimleri.xlsx"),
}

PLAN_DATE = "2026-04-28"  # Excel filtreleme tarihi ÖRNEK 15 NİSAN DEMEK
SERVICE_TIME_MIN = 15
DISTANCE_TO_KM_FACTOR = 0.001
SECONDARY_OBJECTIVE = "cost"  # "cost" veya "distance"
USE_ORDER_DUE_AS_LATEST_TIME = True

# Aynı fiziksel araç için kaçıncı tura kadar sanal slot açılacağı.
# Gerekiyorsa artırılabilir; algoritma ek turları ağır cezalandırdığı için
# yeterli ilk tur varsa bunlara gitmez.
MAX_TOURS_PER_PHYSICAL_VEHICLE = 3

# Google Routes API. 
GOOGLE_MAPS_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY", "PUT_YOUR_GOOGLE_MAPS_API_KEY_HERE")
GOOGLE_ROUTES_BASE = "https://routes.googleapis.com/directions/v2:computeRoutes"
GOOGLE_TRAVEL_MODE = "DRIVE"
GOOGLE_TIMEOUT = 30
GOOGLE_DEPARTURE_HHMM = "09:00"
DISTANCE_MATRIX_CACHE_FILE = os.path.join(BASE_DIR, "google_distance_matrix_cache.xlsx")
FORCE_REBUILD_DISTANCE_MATRIX = False
GOOGLE_PARTIAL_WRITE_EVERY = 250
GOOGLE_REQUEST_SLEEP_SEC = 0.02
GOOGLE_RETRY_COUNT = 4
GOOGLE_RETRY_BACKOFF_SEC = 1.5
GOOGLE_SESSION_POOL_SIZE = 32

# Harita geometrisi için OSRM fallback
OSRM_BASE = "https://router.project-osrm.org"
OSRM_PROFILE = "driving"
OSRM_TIMEOUT = 30

OUTPUT_FILE = os.path.join(BASE_DIR, "sam_spider_real_data_output_28_04.xlsx")
ROUTE_MAP_OUTPUT_HTML = os.path.join(BASE_DIR, "sam_spider_real_data_route_map_28_04.html")
GENERATE_ROUTE_MAP = True
MAP_TILES = "OpenStreetMap"

# SAM/SPIDER search parameters
RANDOM_SEED = 42
UNASSIGNED_REQUEST_PENALTY = 1_000_000.0
VIOLATION_PENALTY_PER_UNIT = 10_000.0
CONSTRUCTION_ATTEMPTS = 3
TOTAL_TIME_LIMIT_SEC = 3336
FINAL_INTENSIFICATION_RESERVE_SEC = 5.0
DESTROY_MIN = 4
DESTROY_MAX = 14
SHAW_WEIGHT_DIST = 1.0
SHAW_WEIGHT_TIME = 0.2
SA_START_TEMP = 1500.0
SA_COOLING = 0.995
MAX_EXCHANGE_PAIRS = 500
FINAL_VND_ROUNDS = 8

# Amaç kaymasını engellemek için gecikme/tekrar tur cezaları
TOUR_SLOT_WEIGHT = 100_000.0            # her kullanılan turun temel cezası
PHYSICAL_VEHICLE_WEIGHT = 10_000.0      # kullanılan fiziksel araç sayısı
REPEAT_TOUR_PENALTY = 1_000_000.0       # ilk turdan sonraki her ek tur çok pahalı
NEXT_DAY_SERVICE_PENALTY = 2_000_000.0  # ertesi güne sarkan servis cezası
NEXT_DAY_DELAY_PER_MIN = 100.0          # ilk gün kapanışına göre gecikme cezası

CONSTRAINT_GROUP_ORDER = [
    "request_assign",
    "capacity",
    "time_window",
    "start_end",
    "compat",
    "repeat",
]


# =========================
# COMMON HELPERS
# =========================
def safe_round(x, nd=4):
    if x is None or x == "":
        return None if x is None else ""
    return round(float(x), nd)


def norm_str(x):
    if pd.isna(x):
        return ""
    return str(x).strip().replace("\xa0", " ").upper()


def clean_numeric_value(value, field_name="numeric", allow_empty=False):
    if pd.isna(value) or value == "":
        if allow_empty:
            return 0.0
        raise ValueError(f"{field_name} bos veya okunamadi: {value!r}")
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)
    text = str(value).strip().replace("\xa0", " ")
    text = text.replace("−", "-").replace("–", "-")
    text = text.strip(" ;")
    while text.endswith((",", ".")):
        text = text[:-1].strip()
    if "," in text and "." in text:
        last_comma = text.rfind(",")
        last_dot = text.rfind(".")
        if last_comma > last_dot:
            text = text.replace(".", "").replace(",", ".")
        else:
            text = text.replace(",", "")
    elif "," in text:
        parts = [x.strip() for x in text.split(",") if x.strip() != ""]
        if len(parts) == 1:
            text = parts[0]
        elif len(parts) == 2 and parts[0].replace("-", "", 1).isdigit() and parts[1].isdigit():
            text = parts[0] + "." + parts[1]
        else:
            text = parts[0]
    text = text.replace(" ", "")
    try:
        return float(text)
    except Exception as exc:
        raise ValueError(f"{field_name} sayiya cevrilemedi: {value!r}") from exc


def coerce_numeric_series(series, field_name, allow_empty=False):
    return series.map(lambda x: clean_numeric_value(x, field_name=field_name, allow_empty=allow_empty)).astype(float)


def extract_numbers_from_text(value):
    if pd.isna(value):
        return []
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return [float(value)]
    text = str(value).strip().replace("\xa0", " ").replace("−", "-").replace("–", "-")
    import re
    return [float(m.group(0).replace(",", ".")) for m in re.finditer(r"[-+]?\d+(?:[\.,]\d+)?", text)]


def clean_coordinates_df(loc_df):
    df = loc_df.copy()
    validate_columns(df, ["Lokasyon_Kodu", "Enlem", "Boylam"], "Lokasyon bilgileri")
    cleaned_lat, cleaned_lon = [], []
    warnings = []
    for idx, row in df.iterrows():
        code = row.get("Lokasyon_Kodu", idx)
        lat_nums = extract_numbers_from_text(row.get("Enlem"))
        lon_nums = extract_numbers_from_text(row.get("Boylam"))
        if not lat_nums or not lon_nums:
            cleaned_lat.append(float("nan"))
            cleaned_lon.append(float("nan"))
            warnings.append(f"{code}: Enlem/Boylam eksik veya okunamadi; aktif talep yoksa matristen atlanacak.")
            continue
        lat = lat_nums[0]
        lon = lon_nums[0]
        if len(lon_nums) >= 2 and abs(lat - lon_nums[0]) < 1e-6:
            lon = lon_nums[1]
            warnings.append(f"{code}: Boylam hucresinde iki koordinat vardi; ikinci deger boylam olarak alindi ({lon}).")
        cleaned_lat.append(float(lat))
        cleaned_lon.append(float(lon))
    df["Enlem"] = cleaned_lat
    df["Boylam"] = cleaned_lon
    if warnings:
        print("Koordinat temizleme uyarilari:")
        for msg in warnings[:10]:
            print(" - " + msg)
        if len(warnings) > 10:
            print(f" - ... {len(warnings)-10} ek uyarı")
    return df


def validate_columns(df, required_cols, name):
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} eksik kolonlar: {missing}")


def parse_clock(value):
    if isinstance(value, dtime):
        return value
    if isinstance(value, datetime):
        return value.time()
    if pd.isna(value):
        raise ValueError("Bos saat alani var.")
    text = str(value).strip()
    for fmt in ["%H:%M:%S", "%H:%M"]:
        try:
            return datetime.strptime(text, fmt).time()
        except ValueError:
            pass
    raise ValueError(f"Saat formati okunamadi: {value}")



def filter_orders_for_plan_date(df, date_col, plan_dt):
    """Siparisleri tek dosyadan PLAN_DATE'e gore filtreler.
    Delivery icin Teslim_Tarihi, pickup icin Iade_Tarihi kullanilir.
    Tarih hucrelerinde saat bilgisi olsa bile sadece tarih kismi esas alinir.
    """
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col], errors="coerce")
    if out[date_col].isna().all():
        raise ValueError(f"{date_col} kolonu tarihe cevrilemedi veya tamamen bos.")
    return out[out[date_col].dt.date == plan_dt].copy()

def minutes_from_midnight(t):
    return float(t.hour * 60 + t.minute + t.second / 60.0)


def minutes_to_hhmm(total_minutes):
    if total_minutes is None or total_minutes == "":
        return ""
    total_minutes = int(round(float(total_minutes))) % (24 * 60)
    return f"{total_minutes // 60:02d}:{total_minutes % 60:02d}"


def minutes_to_day_hhmm(total_minutes):
    if total_minutes is None or total_minutes == "":
        return ""
    total_minutes = float(total_minutes)
    day = int(total_minutes // 1440)
    return f"D{day} {minutes_to_hhmm(total_minutes)}"


def build_departure_datetime(plan_dt, hhmm):
    hh, mm = [int(x) for x in hhmm.split(":")[:2]]
    local_dt = datetime.combine(plan_dt, dtime(hh, mm))
    now_utc = datetime.now(timezone.utc)
    dep_utc = local_dt.replace(tzinfo=timezone.utc)
    if dep_utc <= now_utc:
        dep_utc = now_utc + timedelta(minutes=10)
    return dep_utc


def parse_google_duration_to_minutes(duration_str):
    if not duration_str:
        return 0.0
    s = str(duration_str).strip()
    if not s.endswith("s"):
        raise ValueError(f"Beklenmeyen Google duration formati: {duration_str}")
    return float(s[:-1]) / 60.0


def time_exceeded(deadline):
    return deadline is not None and time.time() >= deadline


def main_search_deadline(total_deadline):
    if total_deadline is None or TOTAL_TIME_LIMIT_SEC is None:
        return total_deadline
    reserve = min(FINAL_INTENSIFICATION_RESERVE_SEC, max(0.0, TOTAL_TIME_LIMIT_SEC * 0.2))
    return max(time.time(), total_deadline - reserve)


def active(group, inactive_groups):
    return group not in set(inactive_groups or [])


def add_violation(detail_rows, checked_counts, scenario_name, group, key, lhs, sense, rhs, enabled, tol=1e-6):
    if not enabled:
        return
    checked_counts[group] += 1
    lhs = float(lhs)
    rhs = float(rhs)
    if sense == "==":
        violation = abs(lhs - rhs)
    elif sense == "<=":
        violation = max(0.0, lhs - rhs)
    elif sense == ">=":
        violation = max(0.0, rhs - lhs)
    else:
        raise ValueError(f"Unknown sense: {sense}")
    if violation > tol:
        detail_rows.append({
            "Scenario": scenario_name,
            "Grup": group,
            "Key": str(key),
            "LHS": safe_round(lhs, 6),
            "Sense": sense,
            "RHS": safe_round(rhs, 6),
            "Violation": safe_round(violation, 6),
        })


# =========================
# GOOGLE ROUTES SESSION + DISTANCE MATRIX CACHE
# =========================
_GOOGLE_SESSION = None


def get_google_session():
    global _GOOGLE_SESSION
    if _GOOGLE_SESSION is not None:
        return _GOOGLE_SESSION
    sess = requests.Session()
    if Retry is not None:
        retry = Retry(
            total=0,
            connect=0,
            read=0,
            status=0,
            backoff_factor=0,
            allowed_methods=frozenset(["POST"]),
        )
        adapter = HTTPAdapter(max_retries=retry, pool_connections=GOOGLE_SESSION_POOL_SIZE, pool_maxsize=GOOGLE_SESSION_POOL_SIZE)
    else:
        adapter = HTTPAdapter(pool_connections=GOOGLE_SESSION_POOL_SIZE, pool_maxsize=GOOGLE_SESSION_POOL_SIZE)
    sess.mount("https://", adapter)
    sess.mount("http://", adapter)
    _GOOGLE_SESSION = sess
    return _GOOGLE_SESSION


def fetch_google_route(lat1, lon1, lat2, lon2, departure_dt):
    if not GOOGLE_MAPS_API_KEY or GOOGLE_MAPS_API_KEY == "PUT_YOUR_GOOGLE_MAPS_API_KEY_HERE":
        raise RuntimeError("Google Maps API key tanimli degil. GOOGLE_MAPS_API_KEY ortam degiskenini set edin.")

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": GOOGLE_MAPS_API_KEY,
        "X-Goog-FieldMask": "routes.duration,routes.staticDuration,routes.distanceMeters,routes.polyline.encodedPolyline",
    }
    payload = {
        "origin": {"location": {"latLng": {"latitude": float(lat1), "longitude": float(lon1)}}},
        "destination": {"location": {"latLng": {"latitude": float(lat2), "longitude": float(lon2)}}},
        "travelMode": GOOGLE_TRAVEL_MODE,
        # Trafik bağımlı süre devre dışı: staticDuration kullanılacak.
        "routingPreference": "TRAFFIC_AWARE",
        "departureTime": departure_dt.isoformat().replace("+00:00", "Z"),
        "computeAlternativeRoutes": False,
    }

    sess = get_google_session()
    last_err = None
    for attempt in range(1, GOOGLE_RETRY_COUNT + 1):
        try:
            resp = sess.post(GOOGLE_ROUTES_BASE, headers=headers, json=payload, timeout=GOOGLE_TIMEOUT)
            if resp.status_code in (429, 500, 502, 503, 504):
                raise RuntimeError(f"Google Routes gecici HTTP hatasi: status={resp.status_code}, body={resp.text[:400]}")
            resp.raise_for_status()
            data = resp.json() if resp.text else {}
            if not data.get("routes"):
                raise RuntimeError(f"Google Routes rota dondurmedi. status={resp.status_code}, body={resp.text[:400]}")
            route = data["routes"][0]
            time.sleep(GOOGLE_REQUEST_SLEEP_SEC)
            return {
                "distance_m": float(route.get("distanceMeters", 0.0)),
                "duration_min": parse_google_duration_to_minutes(route.get("duration")),
                "static_duration_min": parse_google_duration_to_minutes(route.get("staticDuration")),
                "encoded_polyline": route.get("polyline", {}).get("encodedPolyline", ""),
                "source": "GOOGLE",
            }
        except Exception as exc:
            last_err = exc
            if attempt >= GOOGLE_RETRY_COUNT:
                break
            sleep_sec = GOOGLE_RETRY_BACKOFF_SEC * attempt
            print(f"UYARI: Google baglanti/cagri hatasi ({attempt}/{GOOGLE_RETRY_COUNT}) {lat1},{lon1}->{lat2},{lon2}. Tekrar denenecek. Hata: {exc}")
            time.sleep(sleep_sec)
    raise RuntimeError(f"Google Routes baglanti/cagri hatasi, retry sonrasi basarisiz: {last_err}")


def normalize_distance_cache_df(df):
    cols = ["Nereden", "Nereye", "Mesafe", "Sure_Dakika", "Statik_Sure_Dakika", "Polyline", "Kaynak"]
    if df is None or df.empty:
        return pd.DataFrame(columns=cols)
    out = df.copy()
    for c in cols:
        if c not in out.columns:
            out[c] = "" if c in ("Polyline", "Kaynak") else None
    out["Nereden"] = out["Nereden"].map(norm_str)
    out["Nereye"] = out["Nereye"].map(norm_str)
    out["Mesafe"] = pd.to_numeric(out["Mesafe"], errors="coerce")
    out["Sure_Dakika"] = pd.to_numeric(out["Sure_Dakika"], errors="coerce")
    out["Statik_Sure_Dakika"] = pd.to_numeric(out["Statik_Sure_Dakika"], errors="coerce")
    out["Polyline"] = out["Polyline"].fillna("").astype(str)
    out["Kaynak"] = out["Kaynak"].fillna("").astype(str)
    out = out.dropna(subset=["Nereden", "Nereye", "Mesafe", "Sure_Dakika"])
    out = out[cols].copy()
    out = out.drop_duplicates(subset=["Nereden", "Nereye"], keep="last")
    return out


def create_or_load_distance_matrix(loc_df, plan_dt):
    validate_columns(loc_df, ["Lokasyon_Kodu", "Enlem", "Boylam"], "Lokasyon bilgileri")
    df = clean_coordinates_df(loc_df)
    df["Lokasyon_Kodu"] = df["Lokasyon_Kodu"].map(norm_str)
    df = df.dropna(subset=["Enlem", "Boylam"]).copy()

    needed_nodes = sorted(df["Lokasyon_Kodu"].unique())
    needed_pairs = {(i, j) for i in needed_nodes for j in needed_nodes}

    cache_df = pd.DataFrame()
    if os.path.exists(DISTANCE_MATRIX_CACHE_FILE) and not FORCE_REBUILD_DISTANCE_MATRIX:
        try:
            cache_df = normalize_distance_cache_df(pd.read_excel(DISTANCE_MATRIX_CACHE_FILE))
            print(f"Mesafe matrisi cache okundu: {DISTANCE_MATRIX_CACHE_FILE}")
        except Exception as exc:
            print(f"UYARI: Mesafe matrisi cache okunamadi, yeniden/artan sekilde uretilecek. Hata: {exc}")
            cache_df = pd.DataFrame()

    cache_df = normalize_distance_cache_df(cache_df)
    cached_pairs = set(zip(cache_df.get("Nereden", pd.Series(dtype=str)), cache_df.get("Nereye", pd.Series(dtype=str))))
    missing_pairs = sorted(list(needed_pairs - cached_pairs))
    loc_map = {
        row["Lokasyon_Kodu"]: {"lat": float(row["Enlem"]), "lon": float(row["Boylam"])}
        for _, row in df.iterrows()
    }

    if missing_pairs:
        print(f"Google Routes API ile statik mesafe/sure matrisi uretiliyor. Eksik pair sayisi: {len(missing_pairs)}")
        dep_dt = build_departure_datetime(plan_dt, GOOGLE_DEPARTURE_HHMM)
        rows = []
        for idx, (origin, dest) in enumerate(missing_pairs, start=1):
            if origin == dest:
                rows.append({
                    "Nereden": origin,
                    "Nereye": dest,
                    "Mesafe": 0.0,
                    "Sure_Dakika": 0.0,
                    "Statik_Sure_Dakika": 0.0,
                    "Polyline": "",
                    "Kaynak": "SELF",
                })
            else:
                oi, dj = loc_map[origin], loc_map[dest]
                info = fetch_google_route(oi["lat"], oi["lon"], dj["lat"], dj["lon"], dep_dt)
                static_min = float(info.get("static_duration_min", 0.0) or 0.0)
                live_min = float(info.get("duration_min", 0.0) or 0.0)
                use_min = static_min if static_min > 1e-9 else live_min
                rows.append({
                    "Nereden": origin,
                    "Nereye": dest,
                    "Mesafe": info["distance_m"],
                    "Sure_Dakika": use_min,
                    "Statik_Sure_Dakika": static_min if static_min > 1e-9 else use_min,
                    "Polyline": info.get("encoded_polyline", ""),
                    "Kaynak": "GOOGLE",
                })
            if idx % max(GOOGLE_PARTIAL_WRITE_EVERY, 1) == 0:
                partial = pd.concat([cache_df, pd.DataFrame(rows)], ignore_index=True)
                partial = normalize_distance_cache_df(partial)
                partial.to_excel(DISTANCE_MATRIX_CACHE_FILE, index=False)
                print(f"  ... {idx}/{len(missing_pairs)} pair tamamlandi, ara cache yazildi.")
        cache_df = pd.concat([cache_df, pd.DataFrame(rows)], ignore_index=True)
        cache_df = normalize_distance_cache_df(cache_df)
        cache_df.to_excel(DISTANCE_MATRIX_CACHE_FILE, index=False)
        print(f"Mesafe matrisi cache'e yazildi: {DISTANCE_MATRIX_CACHE_FILE}")

    out = cache_df[cache_df["Nereden"].isin(needed_nodes) & cache_df["Nereye"].isin(needed_nodes)].copy()
    out = normalize_distance_cache_df(out)
    still_missing = needed_pairs - set(zip(out["Nereden"], out["Nereye"]))
    if still_missing:
        raise RuntimeError(f"Mesafe matrisi eksik kaldi. Eksik pair sayisi: {len(still_missing)}")
    return out


# =========================
# REAL DATA PREPARATION
# =========================
def validate_integer_series(series, series_name, tol=1e-6):
    vals = pd.to_numeric(series, errors="coerce").fillna(0.0).astype(float)
    if ((vals - vals.round()).abs() > tol).any():
        raise ValueError(f"{series_name} tam sayi olmali.")
    return vals.round().astype(int)


def operation_type(delivery_qty, pickup_qty, eps=1e-6):
    if delivery_qty > eps and pickup_qty > eps:
        return "DELIVERY_AND_PICKUP"
    if delivery_qty > eps:
        return "DELIVERY"
    if pickup_qty > eps:
        return "PICKUP"
    return "VISIT"


def line_detail_to_text(items):
    parts = []
    for item in items or []:
        parts.append(f"{item.get('Urun_Cinsi','')}:{item.get('Miktar','')} adet/{safe_round(item.get('Toplam_Hacim',0),2)} hacim")
    return "; ".join(parts)


def resolve_product_volume(product_code, vol_map):
    p = norm_str(product_code)
    if p in vol_map:
        return vol_map[p]
    candidates = []
    for k, v in vol_map.items():
        kk = norm_str(k)
        if p.startswith(kk + " ") or p.startswith(kk + "-") or p.startswith(kk + "/"):
            candidates.append((len(kk), v, kk))
    if candidates:
        candidates.sort(reverse=True)
        return candidates[0][1]
    return None


def split_volume(total_delivery, total_pickup, max_chunk):
    max_qty = max(float(total_delivery), float(total_pickup), 0.0)
    if max_qty <= 1e-9:
        return []
    n = max(1, int(math.ceil(max_qty / max(max_chunk, 1e-9))))
    chunks = []
    for idx in range(n):
        chunks.append({
            "delivery_volume": float(total_delivery) / n,
            "pickup_volume": float(total_pickup) / n,
            "split_index": idx + 1,
            "split_count": n,
        })
    return chunks


def read_inputs():
    return {k: pd.read_excel(v) for k, v in FILES.items()}


def build_plan_context(raw):
    validate_columns(raw["locations"], ["Lokasyon_Kodu", "Enlem", "Boylam", "Bolge", "Ilk_Teslim_Saati", "Son_Teslim_Saati", "Lokasyon_Tipi"], "Lokasyon bilgileri")
    validate_columns(raw["delivery"], ["Musteri", "Urun_Cinsi", "Teslim_Miktari", "Teslim_Tarihi"], "Delivery")
    validate_columns(raw["pickup"], ["Musteri", "Urun_Cinsi", "Iade_Miktari", "Iade_Tarihi"], "Pickup")

    plan_dt = datetime.strptime(PLAN_DATE, "%Y-%m-%d").date()

    locations = clean_coordinates_df(raw["locations"])
    locations["Lokasyon_Kodu"] = locations["Lokasyon_Kodu"].map(norm_str)
    locations["Lokasyon_Tipi"] = locations["Lokasyon_Tipi"].map(norm_str)

    delivery = raw["delivery"].copy()
    delivery["Musteri"] = delivery["Musteri"].map(norm_str)
    delivery["Urun_Cinsi"] = delivery["Urun_Cinsi"].map(norm_str)
    delivery = filter_orders_for_plan_date(delivery, "Teslim_Tarihi", plan_dt)

    pickup = raw["pickup"].copy()
    pickup["Musteri"] = pickup["Musteri"].map(norm_str)
    pickup["Urun_Cinsi"] = pickup["Urun_Cinsi"].map(norm_str)
    pickup = filter_orders_for_plan_date(pickup, "Iade_Tarihi", plan_dt)

    active_customers = sorted(set(delivery["Musteri"]).union(set(pickup["Musteri"])))
    if not active_customers:
        raise ValueError(f"{PLAN_DATE} icin delivery veya pickup bulunamadi.")

    depot_rows = locations[locations["Lokasyon_Tipi"] == "DEPO"].copy()
    if len(depot_rows) != 1:
        raise ValueError("Tam olarak 1 adet DEPO tanimi olmali.")
    depot = depot_rows.iloc[0]["Lokasyon_Kodu"]

    loc_subset = locations[locations["Lokasyon_Kodu"].isin([depot] + active_customers)].copy()
    missing = set(active_customers) - set(loc_subset["Lokasyon_Kodu"])
    if missing:
        raise ValueError(f"Lokasyon bilgilerinde eksik musteriler: {sorted(missing)}")

    return {
        "plan_dt": plan_dt,
        "locations": locations,
        "locations_active": loc_subset,
        "delivery": delivery,
        "pickup": pickup,
        "active_customers": active_customers,
        "depot": depot,
    }


def build_virtual_vehicle_data(vehicles_df):
    vehicles_df = vehicles_df.copy()
    vehicles_df["Plaka"] = vehicles_df["Plaka"].map(norm_str)
    vehicles_df["Arac_Cinsi"] = vehicles_df["Arac_Cinsi"].map(norm_str)
    vehicles_df["Kapasite"] = coerce_numeric_series(vehicles_df["Kapasite"], "Arac Kapasite")
    vehicles_df["Sabit_Maliyet"] = coerce_numeric_series(vehicles_df["Sabit_Maliyet"], "Sabit_Maliyet", allow_empty=True)
    vehicles_df["Km_Maliyeti"] = coerce_numeric_series(vehicles_df["Km_Maliyeti"], "Km_Maliyeti", allow_empty=True)

    physical_vehicles = vehicles_df["Plaka"].tolist()
    vehicle_type_physical = {row["Plaka"]: row["Arac_Cinsi"] for _, row in vehicles_df.iterrows()}
    capacity_physical = {row["Plaka"]: float(row["Kapasite"]) for _, row in vehicles_df.iterrows()}
    fixed_cost_physical = {row["Plaka"]: float(row["Sabit_Maliyet"] or 0.0) for _, row in vehicles_df.iterrows()}
    km_cost_physical = {row["Plaka"]: float(row["Km_Maliyeti"] or 0.0) for _, row in vehicles_df.iterrows()}

    virtual_vehicles = []
    virtual_to_physical = {}
    tour_no = {}
    physical_to_virtuals = defaultdict(list)
    vehicle_type = {}
    capacity = {}
    km_cost = {}

    for plate in physical_vehicles:
        for t in range(1, MAX_TOURS_PER_PHYSICAL_VEHICLE + 1):
            vid = f"{plate}__T{t}"
            virtual_vehicles.append(vid)
            virtual_to_physical[vid] = plate
            tour_no[vid] = t
            physical_to_virtuals[plate].append(vid)
            vehicle_type[vid] = vehicle_type_physical[plate]
            capacity[vid] = capacity_physical[plate]
            km_cost[vid] = km_cost_physical[plate]

    return {
        "physical_vehicles": physical_vehicles,
        "vehicles": virtual_vehicles,
        "virtual_to_physical": virtual_to_physical,
        "tour_no": tour_no,
        "physical_to_virtuals": dict(physical_to_virtuals),
        "vehicle_type": vehicle_type,
        "capacity": capacity,
        "physical_capacity": capacity_physical,
        "physical_fixed_cost": fixed_cost_physical,
        "physical_km_cost": km_cost_physical,
        "km_cost": km_cost,
    }


def prepare_data(raw, ctx, distance_df):
    validate_columns(raw["restrictions"], ["Bolge", "Kisitli_Arac_Cinsi"], "Kisitli bolgeler")
    validate_columns(raw["product_volume"], ["Urun_Cinsi", "Hacim"], "Urun hacimleri")
    validate_columns(raw["vehicles"], ["Plaka", "Kapasite", "Arac_Cinsi", "Sabit_Maliyet", "Km_Maliyeti"], "Arac bilgileri")
    validate_columns(distance_df, ["Nereden", "Nereye", "Mesafe", "Sure_Dakika"], "Mesafe matrisi")

    plan_dt = ctx["plan_dt"]
    locations = ctx["locations"].copy()
    delivery = ctx["delivery"].copy()
    pickup = ctx["pickup"].copy()
    active_customers = ctx["active_customers"]
    depot = ctx["depot"]

    product_volume = raw["product_volume"].copy()
    product_volume["Urun_Cinsi"] = product_volume["Urun_Cinsi"].map(norm_str)
    product_volume["Hacim"] = coerce_numeric_series(product_volume["Hacim"], "Urun hacmi")
    vol_map = {row["Urun_Cinsi"]: float(row["Hacim"]) for _, row in product_volume.iterrows()}

    delivery["Teslim_Miktari"] = validate_integer_series(delivery["Teslim_Miktari"].fillna(0), "Teslim_Miktari")
    delivery["Birim_Hacim"] = delivery["Urun_Cinsi"].map(lambda x: resolve_product_volume(x, vol_map))
    if delivery["Birim_Hacim"].isna().any():
        missing_products = sorted(delivery.loc[delivery["Birim_Hacim"].isna(), "Urun_Cinsi"].unique())
        raise ValueError(f"Delivery dosyasinda hacmi tanimsiz urun var: {missing_products}")
    delivery["Toplam_Hacim"] = delivery["Teslim_Miktari"] * delivery["Birim_Hacim"]

    pickup["Iade_Miktari"] = validate_integer_series(pickup["Iade_Miktari"].fillna(0), "Iade_Miktari")
    pickup["Birim_Hacim"] = pickup["Urun_Cinsi"].map(lambda x: resolve_product_volume(x, vol_map))
    if pickup["Birim_Hacim"].isna().any():
        missing_products = sorted(pickup.loc[pickup["Birim_Hacim"].isna(), "Urun_Cinsi"].unique())
        raise ValueError(f"Pickup dosyasinda hacmi tanimsiz urun var: {missing_products}")
    pickup["Toplam_Hacim"] = pickup["Iade_Miktari"] * pickup["Birim_Hacim"]

    loc_customers = locations[(locations["Lokasyon_Tipi"] == "MUSTERI") & (locations["Lokasyon_Kodu"].isin(active_customers))].copy()

    depot_row = locations[locations["Lokasyon_Kodu"] == depot].iloc[0]
    depot_start_min = minutes_from_midnight(parse_clock(depot_row["Ilk_Teslim_Saati"]))
    depot_end_min = minutes_from_midnight(parse_clock(depot_row["Son_Teslim_Saati"]))

    tw_open, tw_close, region = {}, {}, {}
    for _, row in loc_customers.iterrows():
        cust = row["Lokasyon_Kodu"]
        open_t = parse_clock(row["Ilk_Teslim_Saati"])
        close_t = parse_clock(row["Son_Teslim_Saati"])
        open_min = minutes_from_midnight(open_t)
        close_min = minutes_from_midnight(close_t)
        if USE_ORDER_DUE_AS_LATEST_TIME:
            due_times = []
            delivery_due = pd.to_datetime(delivery.loc[delivery["Musteri"] == cust, "Teslim_Tarihi"], errors="coerce").dropna()
            pickup_due = pd.to_datetime(pickup.loc[pickup["Musteri"] == cust, "Iade_Tarihi"], errors="coerce").dropna()
            due_times.extend([x.time() for x in delivery_due])
            due_times.extend([x.time() for x in pickup_due])
            if due_times:
                close_min = min(close_min, min(minutes_from_midnight(x) for x in due_times))
        if close_min < open_min:
            raise ValueError(f"{cust} zaman penceresi gecersiz.")
        tw_open[cust] = open_min
        tw_close[cust] = close_min
        region[cust] = norm_str(row["Bolge"])

    veh = build_virtual_vehicle_data(raw["vehicles"])

    restrictions = raw["restrictions"].copy()
    restrictions["Bolge"] = restrictions["Bolge"].map(norm_str)
    restrictions["Kisitli_Arac_Cinsi"] = restrictions["Kisitli_Arac_Cinsi"].map(norm_str)
    restricted_pairs = set(zip(restrictions["Bolge"], restrictions["Kisitli_Arac_Cinsi"]))
    compat = {}
    for cust in active_customers:
        for v in veh["vehicles"]:
            compat[(cust, v)] = 0 if (region[cust], veh["vehicle_type"][v]) in restricted_pairs else 1
        if sum(compat[(cust, v)] for v in veh["vehicles"]) == 0:
            raise ValueError(f"{cust} hicbir arac tipi ile uyumlu degil.")

    del_by_cust = defaultdict(float)
    pick_by_cust = defaultdict(float)
    del_lines_by_cust = defaultdict(list)
    pick_lines_by_cust = defaultdict(list)
    for idx, row in delivery.reset_index(drop=True).iterrows():
        line = {"Line_ID": f"D{idx+1}", "Musteri": row["Musteri"], "Urun_Cinsi": row["Urun_Cinsi"], "Miktar": int(row["Teslim_Miktari"]), "Toplam_Hacim": float(row["Toplam_Hacim"])}
        del_lines_by_cust[row["Musteri"]].append(line)
        del_by_cust[row["Musteri"]] += float(row["Toplam_Hacim"])
    for idx, row in pickup.reset_index(drop=True).iterrows():
        line = {"Line_ID": f"P{idx+1}", "Musteri": row["Musteri"], "Urun_Cinsi": row["Urun_Cinsi"], "Miktar": int(row["Iade_Miktari"]), "Toplam_Hacim": float(row["Toplam_Hacim"])}
        pick_lines_by_cust[row["Musteri"]].append(line)
        pick_by_cust[row["Musteri"]] += float(row["Toplam_Hacim"])

    requests_dict = {}
    service_to_customer = {}
    for cust in active_customers:
        max_compat_capacity = max(veh["capacity"][v] for v in veh["vehicles"] if compat[(cust, v)] == 1 and veh["tour_no"][v] == 1)
        chunks = split_volume(del_by_cust[cust], pick_by_cust[cust], max_compat_capacity)
        for ch in chunks:
            rid = f"R_{cust}_{ch['split_index']:02d}"
            requests_dict[rid] = {
                "request_id": rid,
                "customer": cust,
                "delivery_volume": ch["delivery_volume"],
                "pickup_volume": ch["pickup_volume"],
                "split_index": ch["split_index"],
                "split_count": ch["split_count"],
                "early": tw_open[cust],
                "late": tw_close[cust],
                "service": SERVICE_TIME_MIN,
                "latest_allowed_abs": 2880.0 + tw_close[cust],  # en geç ertesi gün aynı pencere kapanışı
            }
            service_to_customer[rid] = cust

    nodes = [depot] + active_customers
    dist_df = distance_df.copy()
    dist_df["Nereden"] = dist_df["Nereden"].map(norm_str)
    dist_df["Nereye"] = dist_df["Nereye"].map(norm_str)
    dist_df["Mesafe"] = coerce_numeric_series(dist_df["Mesafe"], "Mesafe")
    dist_df["Sure_Dakika"] = coerce_numeric_series(dist_df["Sure_Dakika"], "Sure_Dakika")
    if "Statik_Sure_Dakika" in dist_df.columns:
        dist_df["Statik_Sure_Dakika"] = coerce_numeric_series(dist_df["Statik_Sure_Dakika"], "Statik_Sure_Dakika", allow_empty=True)
    else:
        dist_df["Statik_Sure_Dakika"] = dist_df["Sure_Dakika"]

    distance, travel_time, polyline = {}, {}, {}
    for i in nodes:
        for j in nodes:
            if i == j:
                distance[(i, j)] = 0.0
                travel_time[(i, j)] = 0.0
                polyline[(i, j)] = ""
                continue
            match = dist_df[(dist_df["Nereden"] == i) & (dist_df["Nereye"] == j)]
            if match.empty:
                raise ValueError(f"Mesafe matrisi eksik: {i} -> {j}")
            row = match.iloc[0]
            distance[(i, j)] = float(row["Mesafe"])
            travel_time[(i, j)] = float(row["Statik_Sure_Dakika"] if float(row.get("Statik_Sure_Dakika", 0.0) or 0.0) > 1e-9 else row["Sure_Dakika"])
            polyline[(i, j)] = str(row.get("Polyline", "") or "")

    coord_map = {}
    for _, row in locations[locations["Lokasyon_Kodu"].isin(nodes)].iterrows():
        coord_map[row["Lokasyon_Kodu"]] = {
            "lat": float(row["Enlem"]),
            "lon": float(row["Boylam"]),
            "bolge": norm_str(row["Bolge"]),
            "lokasyon_tipi": norm_str(row["Lokasyon_Tipi"]),
        }

    return {
        "instance_name": f"REAL_{PLAN_DATE}",
        "plan_date": plan_dt,
        "depot": depot,
        "customers": active_customers,
        "physical_vehicles": veh["physical_vehicles"],
        "vehicles": veh["vehicles"],  # sanal tur slotları
        "virtual_to_physical": veh["virtual_to_physical"],
        "tour_no": veh["tour_no"],
        "physical_to_virtuals": veh["physical_to_virtuals"],
        "max_vehicles": len(veh["physical_vehicles"]),
        "capacity": veh["capacity"],
        "vehicle_type": veh["vehicle_type"],
        "physical_capacity": veh["physical_capacity"],
        "physical_fixed_cost": veh["physical_fixed_cost"],
        "physical_km_cost": veh["physical_km_cost"],
        "km_cost": veh["km_cost"],
        "compat": compat,
        "region": region,
        "requests": requests_dict,
        "service_to_customer": service_to_customer,
        "distance": distance,
        "travel_time": travel_time,
        "depot_start_min": depot_start_min,
        "depot_end_min": depot_end_min,
        "polyline": polyline,
        "tw_open": tw_open,
        "tw_close": tw_close,
        "coord_map": coord_map,
        "delivery_lines_by_customer": dict(del_lines_by_cust),
        "pickup_lines_by_customer": dict(pick_lines_by_cust),
        "total_delivery_by_customer": defaultdict(float, del_by_cust),
        "total_pickup_by_customer": defaultdict(float, pick_by_cust),
    }


# =========================
# SAM/SPIDER CORE ADAPTED TO MULTI-TOUR REAL SERVICE REQUESTS
# =========================
def empty_state(data):
    depot = data["depot"]
    return {
        "routes": {v: [depot, depot] for v in data["vehicles"]},
        "request_vehicle": {},
        "unassigned_requests": set(data["requests"].keys()),
    }


def clone_state(state):
    return {
        "routes": {k: list(v) for k, v in state["routes"].items()},
        "request_vehicle": dict(state["request_vehicle"]),
        "unassigned_requests": set(state["unassigned_requests"]),
    }


def route_location(node, data):
    return data["depot"] if node == data["depot"] else data["requests"][node]["customer"]


def route_distance(route, data):
    total = 0.0
    for i in range(len(route) - 1):
        a = route_location(route[i], data)
        b = route_location(route[i + 1], data)
        total += data["distance"][(a, b)]
    return total


def route_variable_cost(route, data, vehicle):
    km = route_distance(route, data) * DISTANCE_TO_KM_FACTOR
    return km * data["km_cost"].get(vehicle, 0.0)



def physical_has_other_used_tours(state, data, vehicle):
    physical = data["virtual_to_physical"][vehicle]
    for vv in data["physical_to_virtuals"][physical]:
        if vv == vehicle:
            continue
        if len(normalize_vehicle_route(state["routes"].get(vv), data)) > 2:
            return True
    return False


def route_search_cost(route, data, vehicle, state=None):
    if str(SECONDARY_OBJECTIVE).lower() == "cost":
        val = route_variable_cost(route, data, vehicle)
        if state is not None and len(route) > 2 and not physical_has_other_used_tours(state, data, vehicle):
            val += data["physical_fixed_cost"].get(data["virtual_to_physical"][vehicle], 0.0)
        return val
    return route_distance(route, data)


def route_request_ids(route, data):
    return [n for n in route if n != data["depot"]]


def normalize_vehicle_route(route, data):
    route = list(route or [data["depot"], data["depot"]])
    if not route:
        route = [data["depot"], data["depot"]]
    if route[0] != data["depot"]:
        route = [data["depot"]] + route
    if route[-1] != data["depot"]:
        route = route + [data["depot"]]
    if len(route) < 2:
        route = [data["depot"], data["depot"]]
    return route


def simulate_physical_vehicle_chain(state, data, physical, inactive_groups=None, route_overrides=None):
    """Aynı fiziksel araca ait sanal turları zincir halinde simüle eder."""
    inactive_groups = set(inactive_groups or [])
    route_overrides = route_overrides or {}
    contexts = {}

    prev_used = False
    prev_return = None

    for idx, v in enumerate(data["physical_to_virtuals"][physical], start=1):
        route = normalize_vehicle_route(route_overrides.get(v, state["routes"].get(v)), data)
        is_used = len(route) > 2
        start_time = float(data["depot_start_min"])
        blocked_reason = None
        er = None

        if idx > 1:
            if not prev_used:
                blocked_reason = "repeat_order"
            elif prev_return is None:
                blocked_reason = "repeat_prev_infeasible"
            else:
                start_time = float(prev_return)
                if start_time > data["depot_end_min"] + 1e-9:
                    blocked_reason = "repeat_depot_close"

        if blocked_reason is None:
            er = evaluate_route(route, data, v, start_time=start_time, inactive_groups=inactive_groups)
            if not er["feasible"]:
                blocked_reason = er.get("reason", "route_infeasible")

        contexts[v] = {
            "physical_vehicle": physical,
            "tour_no": data["tour_no"][v],
            "route": route,
            "is_used": is_used,
            "start_time": start_time,
            "blocked_reason": blocked_reason,
            "return_time": er.get("return_time") if er and er.get("feasible") else None,
            "detail": er.get("detail", []) if er else [],
            "initial_load": er.get("initial_load", 0.0) if er else 0.0,
            "feasible_route": bool(er and er.get("feasible")),
        }

        prev_used = is_used
        prev_return = contexts[v]["return_time"] if is_used and contexts[v]["feasible_route"] else None

    return contexts


def get_vehicle_slot_context(state, data, vehicle, inactive_groups=None, route_overrides=None):
    physical = data["virtual_to_physical"][vehicle]
    return simulate_physical_vehicle_chain(
        state,
        data,
        physical,
        inactive_groups=inactive_groups,
        route_overrides=route_overrides,
    )[vehicle]


def compute_customer_service_start(arrival, req):
    """Müşteri en geç ikinci gün hizmet alabilir."""
    open0 = req["early"]
    close0 = req["late"]
    open1 = 1440.0 + open0
    close1 = 1440.0 + close0
    open2 = 2880.0 + open0
    close2 = 2880.0 + close0

    # Aynı gün pencere içinde hizmet
    if arrival <= close0 + 1e-9:
        start = max(arrival, open0)
        if start <= close0 + 1e-9:
            return start, 0

    # Ertesi gün pencere içinde hizmet
    if arrival <= close1 + 1e-9:
        start = max(arrival, open1)
        if start <= close1 + 1e-9:
            return start, 1

    # İkinci gün pencere içinde hizmet
    if arrival <= close2 + 1e-9:
        start = max(arrival, open2)
        if start <= close2 + 1e-9:
            return start, 2

    return None, None

def evaluate_route(route, data, vehicle, start_time=None, inactive_groups=None):
    inactive_groups = set(inactive_groups or [])
    depot = data["depot"]
    if route[0] != depot or route[-1] != depot:
        if active("start_end", inactive_groups):
            return {"feasible": False, "reason": "start_end", "detail": []}

    if start_time is None:
        start_time = float(data.get("depot_start_min", 0.0))

    initial_load = sum(data["requests"][rid]["delivery_volume"] for rid in route if rid != depot)
    if active("capacity", inactive_groups) and initial_load > data["capacity"][vehicle] + 1e-9:
        return {"feasible": False, "reason": f"capacity_initial_{vehicle}", "detail": []}

    current_load = initial_load
    current_time = float(start_time)
    prev_loc = depot
    detail = []

    for node in route[1:]:
        loc = route_location(node, data)
        departure_time = current_time
        travel = data["travel_time"][(prev_loc, loc)]
        arrival = current_time + travel

        if node == depot:
            service_start = arrival
            service_end = arrival
            detail.append({
                "node": node,
                "customer": depot,
                "departure_time": departure_time,
                "arrival": arrival,
                "service_start": service_start,
                "service_end": service_end,
                "load_after": current_load,
                "travel": travel,
                "static_travel": travel,
                "delivery_volume": 0.0,
                "pickup_volume": 0.0,
                "service_day": int(service_start // 1440),
                "next_day_service": 0,
                "delay_from_day0_close": 0.0,
                "wait_before_service": max(0.0, service_start - arrival),
            })
            current_time = service_end
            prev_loc = loc
            continue

        req = data["requests"][node]
        cust = req["customer"]
        if active("compat", inactive_groups) and data["compat"][(cust, vehicle)] != 1:
            return {"feasible": False, "reason": f"compat_{node}_{vehicle}", "detail": detail}

        service_start, service_day = compute_customer_service_start(arrival, req)
        if service_start is None:
            return {"feasible": False, "reason": f"time_window_{node}", "detail": detail}

        if active("time_window", inactive_groups) and service_start > req["latest_allowed_abs"] + 1e-9:
            return {"feasible": False, "reason": f"time_window_{node}", "detail": detail}

        current_load = current_load - req["delivery_volume"] + req["pickup_volume"]
        if active("capacity", inactive_groups) and (current_load < -1e-9 or current_load > data["capacity"][vehicle] + 1e-9):
            return {"feasible": False, "reason": f"capacity_{node}", "detail": detail}

        service_end = service_start + req["service"]
        detail.append({
            "node": node,
            "customer": cust,
            "departure_time": departure_time,
            "arrival": arrival,
            "service_start": service_start,
            "service_end": service_end,
            "load_after": current_load,
            "travel": travel,
            "static_travel": travel,
            "delivery_volume": req["delivery_volume"],
            "pickup_volume": req["pickup_volume"],
            "service_day": service_day,
            "next_day_service": 1 if service_day >= 1 else 0,
            "delay_from_day0_close": max(0.0, service_start - req["late"]),
            "wait_before_service": max(0.0, service_start - arrival),
        })
        current_time = service_end
        prev_loc = loc

    return {
        "feasible": True,
        "detail": detail,
        "return_time": current_time,
        "initial_load": initial_load,
        "start_time": start_time,
    }


def search_score(used_tours, used_physical, repeat_tours, next_day_count, next_day_delay_min, secondary_objective_value):
    return (
        next_day_count * NEXT_DAY_SERVICE_PENALTY
        + next_day_delay_min * NEXT_DAY_DELAY_PER_MIN
        + repeat_tours * REPEAT_TOUR_PENALTY
        + used_tours * TOUR_SLOT_WEIGHT
        + used_physical * PHYSICAL_VEHICLE_WEIGHT
        + secondary_objective_value
    )



def evaluate_state(data, state, inactive_groups=None, scenario_name="BASE_MODEL"):
    inactive_groups = set(inactive_groups or [])
    detail_rows = []
    checked_counts = defaultdict(int)
    request_assign_counts = defaultdict(int)
    route_metrics = {}
    used_virtual = 0
    used_physical_set = set()
    repeat_tour_count = 0
    next_day_service_count = 0
    next_day_delay_min = 0.0

    for physical in data["physical_vehicles"]:
        physical_ctx = simulate_physical_vehicle_chain(state, data, physical, inactive_groups=inactive_groups)
        first_used_v = None

        for v in data["physical_to_virtuals"][physical]:
            ctx = physical_ctx[v]
            route = ctx["route"]
            is_used = ctx["is_used"]
            detail = ctx.get("detail", [])
            blocked_reason = ctx.get("blocked_reason")
            feasible_route = bool(ctx.get("feasible_route"))

            add_violation(detail_rows, checked_counts, scenario_name, "start_end", f"{v}_start", 1 if route[0] == data["depot"] else 0, "==", 1, active("start_end", inactive_groups))
            add_violation(detail_rows, checked_counts, scenario_name, "start_end", f"{v}_end", 1 if route[-1] == data["depot"] else 0, "==", 1, active("start_end", inactive_groups))

            if is_used and not feasible_route:
                reason = blocked_reason or "unknown"
                if str(reason).startswith("capacity"):
                    add_violation(detail_rows, checked_counts, scenario_name, "capacity", f"{v}_{reason}", 1, "==", 0, active("capacity", inactive_groups))
                elif str(reason).startswith("time_window"):
                    add_violation(detail_rows, checked_counts, scenario_name, "time_window", f"{v}_{reason}", 1, "==", 0, active("time_window", inactive_groups))
                elif str(reason).startswith("compat"):
                    add_violation(detail_rows, checked_counts, scenario_name, "compat", f"{v}_{reason}", 1, "==", 0, active("compat", inactive_groups))
                elif str(reason).startswith("repeat"):
                    add_violation(detail_rows, checked_counts, scenario_name, "repeat", f"{v}_{reason}", 1, "==", 0, active("repeat", inactive_groups))
                elif str(reason).startswith("start_end"):
                    add_violation(detail_rows, checked_counts, scenario_name, "start_end", f"{v}_{reason}", 1, "==", 0, active("start_end", inactive_groups))

            for rid in route_request_ids(route, data):
                request_assign_counts[rid] += 1

            dist_val = route_distance(route, data)
            var_cost_val = route_variable_cost(route, data, v) if is_used else 0.0
            route_metrics[v] = {
                "route": route,
                "used": is_used,
                "distance": dist_val,
                "optimization_cost": route_search_cost(route, data, v, state),
                "variable_cost": var_cost_val,
                "fixed_cost": 0.0,
                "total_cost": var_cost_val,
                "detail": detail,
                "feasible_route": feasible_route,
                "blocked_reason": blocked_reason,
                "return_time": ctx.get("return_time", 0.0) or 0.0,
                "initial_load": ctx.get("initial_load", 0.0),
                "start_time": ctx.get("start_time", data["depot_start_min"]),
                "tour_no": data["tour_no"][v],
                "physical_vehicle": physical,
            }

            if is_used:
                used_virtual += 1
                used_physical_set.add(physical)
                if first_used_v is None:
                    first_used_v = v
                else:
                    repeat_tour_count += 1
                if feasible_route:
                    for st in detail:
                        if st["node"] != data["depot"]:
                            next_day_service_count += int(st.get("next_day_service", 0) or 0)
                            next_day_delay_min += float(st.get("delay_from_day0_close", 0.0) or 0.0)

        if first_used_v is not None:
            route_metrics[first_used_v]["fixed_cost"] = data["physical_fixed_cost"].get(physical, 0.0)
            route_metrics[first_used_v]["total_cost"] = route_metrics[first_used_v]["variable_cost"] + route_metrics[first_used_v]["fixed_cost"]

    for rid in data["requests"]:
        assigned = request_assign_counts.get(rid, 0)
        add_violation(detail_rows, checked_counts, scenario_name, "request_assign", rid, assigned, "==", 1, active("request_assign", inactive_groups))

    detail_df = pd.DataFrame(detail_rows)
    summary_rows = []
    for g in CONSTRAINT_GROUP_ORDER:
        gdf = detail_df[detail_df["Grup"] == g] if not detail_df.empty else pd.DataFrame()
        summary_rows.append({
            "Grup": g,
            "Checked_Count": checked_counts.get(g, 0),
            "Violated_Count": int(len(gdf)),
            "Max_Violation": safe_round(gdf["Violation"].max(), 6) if len(gdf) else 0.0,
            "Avg_Violation": safe_round(gdf["Violation"].mean(), 6) if len(gdf) else 0.0,
            "Result": "FAIL" if len(gdf) else "PASS",
        })
    summary_df = pd.DataFrame(summary_rows)

    total_distance = sum(route_metrics[v]["distance"] for v in data["vehicles"] if route_metrics[v]["used"])
    total_variable_cost = sum(route_metrics[v]["variable_cost"] for v in data["vehicles"] if route_metrics[v]["used"])
    total_fixed_cost = sum(route_metrics[v]["fixed_cost"] for v in data["vehicles"] if route_metrics[v]["used"])
    total_transport_cost = total_fixed_cost + total_variable_cost
    secondary_value = total_transport_cost if str(SECONDARY_OBJECTIVE).lower() == "cost" else total_distance
    feasible = detail_df.empty
    unassigned_count = len(state.get("unassigned_requests", set()))
    total_violation = float(detail_df["Violation"].sum()) if not detail_df.empty else 0.0
    used_physical = len(used_physical_set)
    penalized_score = (
        search_score(used_virtual, used_physical, repeat_tour_count, next_day_service_count, next_day_delay_min, secondary_value)
        + UNASSIGNED_REQUEST_PENALTY * unassigned_count
        + VIOLATION_PENALTY_PER_UNIT * total_violation
    )
    return {
        "feasible": feasible,
        "objective": secondary_value,
        "objective_type": str(SECONDARY_OBJECTIVE).lower(),
        "search_score": penalized_score,
        "route_metrics": route_metrics,
        "total_distance": total_distance,
        "total_fixed_cost": total_fixed_cost,
        "total_variable_cost": total_variable_cost,
        "total_transport_cost": total_transport_cost,
        "used_vehicles": used_virtual,
        "used_physical_vehicles": used_physical,
        "repeat_tour_count": repeat_tour_count,
        "next_day_service_count": next_day_service_count,
        "next_day_delay_min": next_day_delay_min,
        "unassigned_count": unassigned_count,
        "total_violation": total_violation,
        "feasibility": {"summary": summary_df, "detail": detail_df},
        "checked_counts": dict(checked_counts),
    }


def insertion_positions(route):
    n = len(route)
    for pos in range(1, n):
        yield pos


def insert_request_into_route(route, rid, pos):
    r = list(route)
    r.insert(pos, rid)
    return r


def vehicle_order_for_construction(data):
    ordered = []
    for tour_idx in range(1, MAX_TOURS_PER_PHYSICAL_VEHICLE + 1):
        for physical in data["physical_vehicles"]:
            ordered.append(data["physical_to_virtuals"][physical][tour_idx - 1])
    return ordered



def can_use_virtual_vehicle(state, data, vehicle, inactive_groups=None):
    ctx = get_vehicle_slot_context(state, data, vehicle, inactive_groups=inactive_groups)
    return ctx.get("blocked_reason") is None


def candidate_preserves_vehicle_chain(state, data, vehicle, candidate_route, inactive_groups=None):
    physical = data["virtual_to_physical"][vehicle]
    contexts = simulate_physical_vehicle_chain(
        state,
        data,
        physical,
        inactive_groups=inactive_groups,
        route_overrides={vehicle: candidate_route},
    )
    for vv in data["physical_to_virtuals"][physical]:
        ctx = contexts[vv]
        if ctx["is_used"] and (ctx["blocked_reason"] is not None or not ctx["feasible_route"]):
            return False, contexts
    return True, contexts


def best_insertion_for_request(state, data, rid, inactive_groups=None, vehicle_subset=None, deadline=None):
    best = None
    vehicles = vehicle_subset if vehicle_subset is not None else vehicle_order_for_construction(data)
    req = data["requests"][rid]
    slot_contexts = compute_virtual_slot_contexts(state, data, inactive_groups=inactive_groups)

    for v in vehicles:
        if time_exceeded(deadline):
            break
        slot_ctx = slot_contexts.get(v, {})
        if slot_ctx.get("blocked_reason") is not None:
            continue
        if active("compat", inactive_groups) and data["compat"][(req["customer"], v)] != 1:
            continue

        base_route = normalize_vehicle_route(state["routes"].get(v), data)
        base_cost = route_search_cost(base_route, data, v, state)

        for pos in insertion_positions(base_route):
            if time_exceeded(deadline):
                break
            candidate_route = insert_request_into_route(base_route, rid, pos)
            ok, chain_ctx = candidate_preserves_vehicle_chain(state, data, v, candidate_route, inactive_groups=inactive_groups)
            if not ok:
                continue

            delta = route_search_cost(candidate_route, data, v, state) - base_cost
            slack = max(0.0, req["late"] - req["early"])
            repeat_bias = (data["tour_no"][v] - 1) * 1000.0
            start_bias = max(0.0, float(chain_ctx[v].get("start_time", data["depot_start_min"])) - float(data["depot_start_min"])) * 1e-3
            crit = delta + 1e-6 * slack + repeat_bias + start_bias
            if best is None or crit < best[0] - 1e-12 or (abs(crit - best[0]) <= 1e-12 and delta < best[1]):
                best = (crit, delta, v, candidate_route)
    return best



def compute_virtual_slot_contexts(state, data, inactive_groups=None):
    """Her sanal tur slotu için zincirsel başlangıç zamanı ve blokaj sebebini hesaplar."""
    inactive_groups = set(inactive_groups or [])
    ctx = {}
    for physical in data["physical_vehicles"]:
        ctx.update(simulate_physical_vehicle_chain(state, data, physical, inactive_groups=inactive_groups))
    return ctx



def summarize_reason_counts(reason_counts):
    if not reason_counts:
        return ""
    items = sorted(reason_counts.items(), key=lambda x: (-x[1], str(x[0])))
    return "; ".join(f"{k}:{v}" for k, v in items)


def infer_main_cause(req, current_feasible_insertions, standalone, reason_counts, compat_slots, usable_slots):
    if current_feasible_insertions > 0:
        return "search_stuck_despite_feasible_insertion"
    if compat_slots <= 0:
        return "no_compatible_vehicle"
    if usable_slots <= 0:
        if reason_counts.get("repeat_order", 0) > 0 or reason_counts.get("repeat_prev_infeasible", 0) > 0 or reason_counts.get("repeat_depot_close", 0) > 0:
            ranked = sorted(reason_counts.items(), key=lambda x: (-x[1], str(x[0])))
            return ranked[0][0] if ranked else "no_usable_tour_slot"
        return "no_usable_tour_slot"
    if not standalone.get("feasible_any", False):
        ranked = sorted(reason_counts.items(), key=lambda x: (-x[1], str(x[0])))
        return ranked[0][0] if ranked else "standalone_infeasible"
    ranked = sorted(reason_counts.items(), key=lambda x: (-x[1], str(x[0])))
    return ranked[0][0] if ranked else "insertion_failed"


def standalone_request_feasibility(data, rid, inactive_groups=None):
    """Atanmamış bir request'in boş sistemde ilk turda tek başına servislenebilirliğini test eder."""
    inactive_groups = set(inactive_groups or [])
    req = data["requests"][rid]
    depot = data["depot"]
    rows = []
    feasible_any = False
    best_vehicle = ""
    best_service_start = None

    for physical in data["physical_vehicles"]:
        v = data["physical_to_virtuals"][physical][0]
        compatible = (not active("compat", inactive_groups)) or (data["compat"][(req["customer"], v)] == 1)
        if not compatible:
            rows.append({
                "Vehicle": v,
                "Physical_Vehicle": physical,
                "Tour_No": data["tour_no"][v],
                "Compatible": "NO",
                "Feasible": "NO",
                "Reason": "compat",
                "Service_Start": "",
                "Return_Time": "",
            })
            continue

        route = [depot, rid, depot]
        er = evaluate_route(route, data, v, start_time=float(data.get("depot_start_min", 0.0)), inactive_groups=inactive_groups)
        if er["feasible"]:
            feasible_any = True
            service_start = None
            for stop in er.get("detail", []):
                if stop.get("node") == rid:
                    service_start = stop.get("service_start")
                    break
            if best_service_start is None or (service_start is not None and service_start < best_service_start):
                best_service_start = service_start
                best_vehicle = v
            rows.append({
                "Vehicle": v,
                "Physical_Vehicle": physical,
                "Tour_No": data["tour_no"][v],
                "Compatible": "YES",
                "Feasible": "YES",
                "Reason": "",
                "Service_Start": minutes_to_hhmm(service_start) if service_start is not None else "",
                "Return_Time": minutes_to_hhmm(er.get("return_time")) if er.get("return_time") is not None else "",
            })
        else:
            rows.append({
                "Vehicle": v,
                "Physical_Vehicle": physical,
                "Tour_No": data["tour_no"][v],
                "Compatible": "YES",
                "Feasible": "NO",
                "Reason": er.get("reason", "unknown"),
                "Service_Start": "",
                "Return_Time": "",
            })

    return {
        "feasible_any": feasible_any,
        "best_vehicle": best_vehicle,
        "best_service_start": best_service_start,
        "rows": rows,
    }


def diagnose_infeasibility(data, state, eval_result, inactive_groups=None, model_name="SAM_SPIDER_REAL_SD_HF_COMPAT_REPEATABLE", stop_reason="", iteration_count=0):
    inactive_groups = set(inactive_groups or [])
    slot_ctx = compute_virtual_slot_contexts(state, data, inactive_groups=inactive_groups)
    unassigned = sorted(list(state.get("unassigned_requests", set())))

    summary_rows = [{
        "Scenario": model_name,
        "StopReason": stop_reason,
        "IterationsDone": iteration_count,
        "Feasible": "YES" if eval_result.get("feasible") else "NO",
        "Unassigned_Request_Count": len(unassigned),
        "Used_Tour_Slots": eval_result.get("used_vehicles", 0),
        "Used_Physical_Vehicles": eval_result.get("used_physical_vehicles", 0),
        "Repeat_Tour_Count": eval_result.get("repeat_tour_count", 0),
        "Next_Day_Service_Count": eval_result.get("next_day_service_count", 0),
        "Next_Day_Delay_Min": safe_round(eval_result.get("next_day_delay_min", 0.0), 2),
        "Total_Violation": safe_round(eval_result.get("total_violation", 0.0), 4),
        "Objective_Type": eval_result.get("objective_type", SECONDARY_OBJECTIVE),
        "Objective_Value": safe_round(eval_result.get("objective", 0.0), 4),
        "SearchScore": safe_round(eval_result.get("search_score", 0.0), 4),
        "Depot_Open_Min": safe_round(data.get("depot_start_min", 0.0), 2),
        "Depot_Close_Min": safe_round(data.get("depot_end_min", 0.0), 2),
        "Depot_Open_HHMM": minutes_to_hhmm(data.get("depot_start_min", 0.0)),
        "Depot_Close_HHMM": minutes_to_hhmm(data.get("depot_end_min", 0.0)),
    }]

    req_rows = []
    vehicle_rows = []

    for rid in unassigned:
        req = data["requests"][rid]
        standalone = standalone_request_feasibility(data, rid, inactive_groups=inactive_groups)
        reason_counts = defaultdict(int)
        current_feasible_insertions = 0
        compat_slots = 0
        usable_slots = 0

        for v in vehicle_order_for_construction(data):
            base_route = list(state["routes"].get(v, [data["depot"], data["depot"]]))
            req_cust = req["customer"]

            if active("compat", inactive_groups) and data["compat"][(req_cust, v)] != 1:
                vehicle_rows.append({
                    "Request_ID": rid,
                    "Customer": req_cust,
                    "Vehicle": v,
                    "Physical_Vehicle": data["virtual_to_physical"][v],
                    "Tour_No": data["tour_no"][v],
                    "Compatible": "NO",
                    "Usable_Slot": "NO",
                    "Attempted_Positions": 0,
                    "Feasible_Positions": 0,
                    "Top_Reason": "compat",
                    "Reason_Counts": "compat:1",
                })
                reason_counts["compat"] += 1
                continue

            compat_slots += 1
            ctx = slot_ctx.get(v, {})
            blocked_reason = ctx.get("blocked_reason")
            if blocked_reason:
                vehicle_rows.append({
                    "Request_ID": rid,
                    "Customer": req_cust,
                    "Vehicle": v,
                    "Physical_Vehicle": data["virtual_to_physical"][v],
                    "Tour_No": data["tour_no"][v],
                    "Compatible": "YES",
                    "Usable_Slot": "NO",
                    "Attempted_Positions": 0,
                    "Feasible_Positions": 0,
                    "Top_Reason": blocked_reason,
                    "Reason_Counts": f"{blocked_reason}:1",
                })
                reason_counts[blocked_reason] += 1
                continue

            usable_slots += 1
            attempted_positions = 0
            feasible_positions = 0
            local_reasons = defaultdict(int)
            start_time = ctx.get("start_time", data["depot_start_min"])

            for pos in insertion_positions(base_route):
                candidate_route = insert_request_into_route(base_route, rid, pos)
                er = evaluate_route(candidate_route, data, v, start_time=start_time, inactive_groups=inactive_groups)
                attempted_positions += 1
                if er["feasible"]:
                    feasible_positions += 1
                    current_feasible_insertions += 1
                else:
                    local_reasons[er.get("reason", "unknown")] += 1
                    reason_counts[er.get("reason", "unknown")] += 1

            top_reason = "feasible_position_exists" if feasible_positions > 0 else (sorted(local_reasons.items(), key=lambda x: (-x[1], x[0]))[0][0] if local_reasons else "no_position")
            vehicle_rows.append({
                "Request_ID": rid,
                "Customer": req_cust,
                "Vehicle": v,
                "Physical_Vehicle": data["virtual_to_physical"][v],
                "Tour_No": data["tour_no"][v],
                "Compatible": "YES",
                "Usable_Slot": "YES",
                "Attempted_Positions": attempted_positions,
                "Feasible_Positions": feasible_positions,
                "Top_Reason": top_reason,
                "Reason_Counts": summarize_reason_counts(local_reasons),
            })

        req_rows.append({
            "Request_ID": rid,
            "Customer": req["customer"],
            "Split_Index": req.get("split_index", ""),
            "Split_Count": req.get("split_count", ""),
            "Delivery_Volume": safe_round(req.get("delivery_volume", 0.0), 4),
            "Pickup_Volume": safe_round(req.get("pickup_volume", 0.0), 4),
            "Window_Open_Min_Day0": safe_round(req.get("early", 0.0), 2),
            "Window_Close_Min_Day0": safe_round(req.get("late", 0.0), 2),
            "Window_Open_Day0_HHMM": minutes_to_hhmm(req.get("early", 0.0)),
            "Window_Close_Day0_HHMM": minutes_to_hhmm(req.get("late", 0.0)),
            "Latest_Allowed_Abs_Min": safe_round(req.get("latest_allowed_abs", 0.0), 2),
            "Latest_Allowed_Abs_HHMM": f"D{int(req.get('latest_allowed_abs',0)//1440)} {minutes_to_hhmm(req.get('latest_allowed_abs',0))}",
            "Compatible_Tour_Slots": compat_slots,
            "Usable_Tour_Slots_In_Current_State": usable_slots,
            "Feasible_Current_Insertions": current_feasible_insertions,
            "Standalone_FirstTour_Feasible": "YES" if standalone.get("feasible_any") else "NO",
            "Standalone_Best_Vehicle": standalone.get("best_vehicle", ""),
            "Standalone_Best_Service_Start": minutes_to_hhmm(standalone.get("best_service_start")) if standalone.get("best_service_start") is not None else "",
            "Main_Cause": infer_main_cause(req, current_feasible_insertions, standalone, reason_counts, compat_slots, usable_slots),
            "Aggregated_Reason_Counts": summarize_reason_counts(reason_counts),
        })

    global_reason_rows = []
    if not eval_result["feasibility"]["detail"].empty:
        g = eval_result["feasibility"]["detail"].groupby("Grup").agg(
            Violated_Count=("Violation", "count"),
            Total_Violation=("Violation", "sum"),
            Max_Violation=("Violation", "max"),
            Avg_Violation=("Violation", "mean"),
        ).reset_index()
        global_reason_rows = g.to_dict("records")

    standalone_rows = []
    for rid in unassigned:
        st = standalone_request_feasibility(data, rid, inactive_groups=inactive_groups)
        for row in st["rows"]:
            standalone_rows.append({
                "Request_ID": rid,
                "Customer": data["requests"][rid]["customer"],
                **row,
            })

    diagnostics = {
        "summary_df": pd.DataFrame(summary_rows),
        "unassigned_df": pd.DataFrame(req_rows),
        "vehicle_attempt_df": pd.DataFrame(vehicle_rows),
        "standalone_df": pd.DataFrame(standalone_rows),
        "global_violation_df": pd.DataFrame(global_reason_rows),
    }
    return diagnostics



def remove_request_from_state(state, data, rid):
    new_state = clone_state(state)
    assigned_vehicle = new_state["request_vehicle"].get(rid)
    if assigned_vehicle:
        route = [n for n in new_state["routes"][assigned_vehicle] if n != rid]
        if route[0] != data["depot"]:
            route = [data["depot"]] + route
        if route[-1] != data["depot"]:
            route = route + [data["depot"]]
        if len(route) < 2:
            route = [data["depot"], data["depot"]]
        new_state["routes"][assigned_vehicle] = route
        del new_state["request_vehicle"][rid]
    new_state["unassigned_requests"].add(rid)
    return new_state


def assign_request_to_vehicle(state, rid, vehicle, route):
    new_state = clone_state(state)
    new_state["routes"][vehicle] = route
    new_state["request_vehicle"][rid] = vehicle
    new_state["unassigned_requests"].discard(rid)
    return new_state


def candidate_vehicles_for_construction(state, data):
    ordered = vehicle_order_for_construction(data)
    return ordered


def compute_regret_insertions(state, data, unassigned, inactive_groups=None, vehicle_candidates=None, deadline=None):
    info = []
    vehicles_to_test = vehicle_candidates if vehicle_candidates is not None else candidate_vehicles_for_construction(state, data)
    for rid in unassigned:
        if time_exceeded(deadline):
            break
        candidates = []
        for v in vehicles_to_test:
            if time_exceeded(deadline):
                break
            best = best_insertion_for_request(state, data, rid, inactive_groups=inactive_groups, vehicle_subset=[v], deadline=deadline)
            if best is not None:
                candidates.append(best)
        if not candidates:
            info.append((float("-inf"), float("inf"), rid, None))
            continue
        candidates.sort(key=lambda x: x[0])
        best = candidates[0]
        second_val = candidates[1][0] if len(candidates) > 1 else best[0] + 10000.0
        regret = second_val - best[0]
        info.append((regret, best[1], rid, best))
    return info


def request_criticality(data, rid):
    req = data["requests"][rid]
    width = max(1.0, req["late"] - req["early"])
    qty = req["delivery_volume"] + req["pickup_volume"]
    return (width, req["late"], -qty)


def global_regret_build_once(data, inactive_groups=None, rng=None, deadline=None):
    rng = rng or random.Random(RANDOM_SEED)
    state = empty_state(data)
    while state["unassigned_requests"] and not time_exceeded(deadline):
        infos = compute_regret_insertions(state, data, list(state["unassigned_requests"]), inactive_groups=inactive_groups, vehicle_candidates=candidate_vehicles_for_construction(state, data), deadline=deadline)
        feasible_infos = [x for x in infos if x[3] is not None]
        if not feasible_infos:
            break
        def _key(x):
            regret, delta, rid, best = x
            return (-regret, delta, request_criticality(data, rid), rng.random() * 1e-6)
        feasible_infos.sort(key=_key)
        _, _, rid, best = feasible_infos[0]
        _, _, vehicle, route = best
        state = assign_request_to_vehicle(state, rid, vehicle, route)
    return state


def sequential_fill_fallback(state, data, inactive_groups=None, rng=None, deadline=None):
    rng = rng or random.Random(RANDOM_SEED)
    progress = True
    while state["unassigned_requests"] and progress and not time_exceeded(deadline):
        progress = False
        ordered = sorted(list(state["unassigned_requests"]), key=lambda r: (request_criticality(data, r), rng.random()))
        for rid in ordered:
            if time_exceeded(deadline):
                break
            best = best_insertion_for_request(state, data, rid, inactive_groups=inactive_groups, vehicle_subset=candidate_vehicles_for_construction(state, data), deadline=deadline)
            if best is None:
                continue
            _, _, vehicle, route = best
            state = assign_request_to_vehicle(state, rid, vehicle, route)
            progress = True
    return state


def build_initial_solution(data, inactive_groups=None, deadline=None):
    rng = random.Random(RANDOM_SEED)
    best_state, best_eval, best_missing = None, None, float("inf")
    for _ in range(CONSTRUCTION_ATTEMPTS):
        if time_exceeded(deadline):
            break
        candidate = global_regret_build_once(data, inactive_groups=inactive_groups, rng=rng, deadline=deadline)
        if candidate["unassigned_requests"] and not time_exceeded(deadline):
            candidate = sequential_fill_fallback(candidate, data, inactive_groups=inactive_groups, rng=rng, deadline=deadline)
        if candidate["unassigned_requests"] and not time_exceeded(deadline):
            repaired = regret_repair(candidate, data, removed=[], inactive_groups=inactive_groups, deadline=deadline)
            if repaired is not None:
                candidate = repaired
        cand_eval = evaluate_state(data, candidate, inactive_groups=inactive_groups)
        if cand_eval["feasible"] and not time_exceeded(deadline):
            candidate, cand_eval = vnd(candidate, data, inactive_groups=inactive_groups, max_rounds=2, deadline=deadline)
            return candidate, cand_eval
        missing = len(candidate["unassigned_requests"])
        if best_state is None or missing < best_missing or (missing == best_missing and cand_eval["search_score"] < best_eval["search_score"]):
            best_state, best_eval, best_missing = candidate, cand_eval, missing
    if best_state is None:
        raise RuntimeError("Baslangic cozum olusturulamadi.")
    return best_state, best_eval


def tour_depletion_pass(state, data, inactive_groups=None, deadline=None):
    current = clone_state(state)
    improved = False
    used_routes = [(v, len(route_request_ids(current["routes"][v], data))) for v in data["vehicles"] if len(current["routes"][v]) > 2]
    used_routes.sort(key=lambda x: (data["tour_no"][x[0]] > 1, x[1]))  # tekrar turları boşaltmaya daha hevesli ol
    base_eval = evaluate_state(data, current, inactive_groups=inactive_groups)
    for victim, _ in used_routes:
        if time_exceeded(deadline):
            break
        reqs = route_request_ids(current["routes"][victim], data)
        if not reqs:
            continue
        trial = clone_state(current)
        for rid in reqs:
            trial = remove_request_from_state(trial, data, rid)
        success = True
        for rid in sorted(reqs, key=lambda r: data["requests"][r]["late"]):
            if time_exceeded(deadline):
                success = False
                break
            best = best_insertion_for_request(
                trial, data, rid,
                inactive_groups=inactive_groups,
                vehicle_subset=[v for v in candidate_vehicles_for_construction(trial, data) if v != victim],
                deadline=deadline
            )
            if best is None:
                success = False
                break
            _, _, veh, route = best
            trial = assign_request_to_vehicle(trial, rid, veh, route)
        if not success:
            continue
        trial_eval = evaluate_state(data, trial, inactive_groups=inactive_groups)
        if trial_eval["feasible"] and trial_eval["search_score"] + 1e-9 < base_eval["search_score"]:
            current, base_eval, improved = trial, trial_eval, True
    return current, base_eval, improved


def route_2opt_improve(route, data, vehicle, inactive_groups=None, deadline=None):
    best_route = list(route)
    best_cost = route_search_cost(best_route, data, vehicle)
    improved = True
    while improved and not time_exceeded(deadline):
        improved = False
        n = len(best_route)
        for i in range(1, n - 2):
            if time_exceeded(deadline):
                break
            for j in range(i + 1, n - 1):
                if time_exceeded(deadline):
                    break
                cand = best_route[:i] + list(reversed(best_route[i:j + 1])) + best_route[j + 1:]
                er = evaluate_route(cand, data, vehicle, inactive_groups=inactive_groups)
                if not er["feasible"]:
                    continue
                cand_cost = route_search_cost(cand, data, vehicle)
                if cand_cost + 1e-9 < best_cost:
                    best_route, best_cost, improved = cand, cand_cost, True
                    break
            if improved:
                break
    return best_route, best_cost


def intra_oropt_improve(route, data, vehicle, inactive_groups=None, max_seg_len=2, deadline=None):
    best_route = list(route)
    best_cost = route_search_cost(best_route, data, vehicle)
    n = len(best_route)
    improved = True
    while improved and not time_exceeded(deadline):
        improved = False
        for seg_len in range(1, max_seg_len + 1):
            if time_exceeded(deadline):
                break
            for i in range(1, n - 1 - seg_len):
                if time_exceeded(deadline):
                    break
                segment = best_route[i:i + seg_len]
                remainder = best_route[:i] + best_route[i + seg_len:]
                for j in range(1, len(remainder)):
                    if time_exceeded(deadline):
                        break
                    if j == i:
                        continue
                    cand = remainder[:j] + segment + remainder[j:]
                    er = evaluate_route(cand, data, vehicle, inactive_groups=inactive_groups)
                    if not er["feasible"]:
                        continue
                    cand_cost = route_search_cost(cand, data, vehicle)
                    if cand_cost + 1e-9 < best_cost:
                        best_route, best_cost, n, improved = cand, cand_cost, len(cand), True
                        break
                if improved:
                    break
            if improved:
                break
    return best_route, best_cost


def improve_intra_routes(state, data, inactive_groups=None, deadline=None):
    current = clone_state(state)
    improved_any = False
    for v in data["vehicles"]:
        if time_exceeded(deadline):
            break
        route = current["routes"][v]
        if len(route) <= 3:
            continue
        r1, _ = route_2opt_improve(route, data, v, inactive_groups=inactive_groups, deadline=deadline)
        r2, c2 = intra_oropt_improve(r1, data, v, inactive_groups=inactive_groups, deadline=deadline)
        if c2 + 1e-9 < route_search_cost(route, data, v):
            current["routes"][v] = r2
            improved_any = True
    return current, improved_any


def best_relocate_move(state, data, inactive_groups=None, deadline=None):
    base_eval = evaluate_state(data, state, inactive_groups=inactive_groups)
    best_state, best_eval = None, None
    for rid in list(state["request_vehicle"].keys()):
        if time_exceeded(deadline):
            break
        stripped = remove_request_from_state(state, data, rid)
        best = best_insertion_for_request(stripped, data, rid, inactive_groups=inactive_groups, deadline=deadline)
        if best is None:
            continue
        _, _, vehicle, route = best
        candidate = assign_request_to_vehicle(stripped, rid, vehicle, route)
        cand_eval = evaluate_state(data, candidate, inactive_groups=inactive_groups)
        if cand_eval["feasible"] and cand_eval["search_score"] + 1e-9 < base_eval["search_score"]:
            if best_eval is None or cand_eval["search_score"] < best_eval["search_score"]:
                best_state, best_eval = candidate, cand_eval
    return best_state, best_eval


def best_exchange_move(state, data, inactive_groups=None, rng=None, deadline=None):
    base_eval = evaluate_state(data, state, inactive_groups=inactive_groups)
    assigned = list(state["request_vehicle"].keys())
    if len(assigned) < 2:
        return None, None
    pairs = []
    for i in range(len(assigned)):
        if time_exceeded(deadline):
            break
        for j in range(i + 1, len(assigned)):
            if state["request_vehicle"][assigned[i]] == state["request_vehicle"][assigned[j]]:
                continue
            pairs.append((assigned[i], assigned[j]))
    if not pairs:
        return None, None
    if len(pairs) > MAX_EXCHANGE_PAIRS:
        rng = rng or random.Random(RANDOM_SEED)
        pairs = rng.sample(pairs, MAX_EXCHANGE_PAIRS)

    best_state, best_eval = None, None
    for rid1, rid2 in pairs:
        if time_exceeded(deadline):
            break
        stripped = remove_request_from_state(state, data, rid1)
        stripped = remove_request_from_state(stripped, data, rid2)
        first = best_insertion_for_request(stripped, data, rid1, inactive_groups=inactive_groups, deadline=deadline)
        if first is None:
            continue
        _, _, veh1, route1 = first
        st1 = assign_request_to_vehicle(stripped, rid1, veh1, route1)
        second = best_insertion_for_request(st1, data, rid2, inactive_groups=inactive_groups, deadline=deadline)
        if second is None:
            continue
        _, _, veh2, route2 = second
        candidate = assign_request_to_vehicle(st1, rid2, veh2, route2)
        cand_eval = evaluate_state(data, candidate, inactive_groups=inactive_groups)
        if cand_eval["feasible"] and cand_eval["search_score"] + 1e-9 < base_eval["search_score"]:
            if best_eval is None or cand_eval["search_score"] < best_eval["search_score"]:
                best_state, best_eval = candidate, cand_eval
    return best_state, best_eval


def vnd(state, data, inactive_groups=None, max_rounds=5, deadline=None):
    current = clone_state(state)
    current_eval = evaluate_state(data, current, inactive_groups=inactive_groups)
    neighborhoods = ["intra", "relocate", "exchange", "deplete"]
    rounds = 0
    rng = random.Random(RANDOM_SEED)
    while rounds < max_rounds and not time_exceeded(deadline):
        k = 0
        improved = False
        while k < len(neighborhoods) and not time_exceeded(deadline):
            name = neighborhoods[k]
            cand_state, cand_eval = None, None
            if name == "intra":
                cand_state, changed = improve_intra_routes(current, data, inactive_groups=inactive_groups, deadline=deadline)
                if changed:
                    cand_eval = evaluate_state(data, cand_state, inactive_groups=inactive_groups)
            elif name == "relocate":
                cand_state, cand_eval = best_relocate_move(current, data, inactive_groups=inactive_groups, deadline=deadline)
            elif name == "exchange":
                cand_state, cand_eval = best_exchange_move(current, data, inactive_groups=inactive_groups, rng=rng, deadline=deadline)
            elif name == "deplete":
                cand_state, cand_eval, changed = tour_depletion_pass(current, data, inactive_groups=inactive_groups, deadline=deadline)
                if not changed:
                    cand_state, cand_eval = None, None
            if cand_state is not None and cand_eval is not None and cand_eval["feasible"] and cand_eval["search_score"] + 1e-9 < current_eval["search_score"]:
                current, current_eval = cand_state, cand_eval
                k = 0
                improved = True
            else:
                k += 1
        if not improved:
            break
        rounds += 1
    return current, current_eval


def random_destroy(state, data, rng):
    assigned = list(state["request_vehicle"].keys())
    if not assigned:
        return clone_state(state), []
    q = min(len(assigned), rng.randint(DESTROY_MIN, min(DESTROY_MAX, len(assigned))))
    removed = rng.sample(assigned, q)
    new_state = clone_state(state)
    for rid in removed:
        new_state = remove_request_from_state(new_state, data, rid)
    return new_state, removed


def worst_destroy(state, data, rng, inactive_groups=None, deadline=None):
    assigned = list(state["request_vehicle"].keys())
    if not assigned:
        return clone_state(state), []
    base_obj = evaluate_state(data, state, inactive_groups=inactive_groups)["search_score"]
    marginals = []
    for rid in assigned:
        if time_exceeded(deadline):
            break
        st = remove_request_from_state(state, data, rid)
        obj = evaluate_state(data, st, inactive_groups=inactive_groups)["search_score"]
        marginals.append((base_obj - obj, rid))
    marginals.sort(reverse=True)
    if not marginals:
        return clone_state(state), []
    q = min(len(marginals), rng.randint(DESTROY_MIN, min(DESTROY_MAX, len(marginals))))
    removed = [rid for _, rid in marginals[:q]]
    new_state = clone_state(state)
    for rid in removed:
        new_state = remove_request_from_state(new_state, data, rid)
    return new_state, removed


def request_relatedness(data, r1, r2):
    c1 = data["requests"][r1]["customer"]
    c2 = data["requests"][r2]["customer"]
    dist = data["distance"][(c1, c2)]
    tw = abs(data["requests"][r1]["early"] - data["requests"][r2]["early"]) + abs(data["requests"][r1]["late"] - data["requests"][r2]["late"])
    return SHAW_WEIGHT_DIST * dist + SHAW_WEIGHT_TIME * tw


def shaw_destroy(state, data, rng):
    assigned = list(state["request_vehicle"].keys())
    if not assigned:
        return clone_state(state), []
    q = min(len(assigned), rng.randint(DESTROY_MIN, min(DESTROY_MAX, len(assigned))))
    seed = rng.choice(assigned)
    related = [(request_relatedness(data, seed, rid), rid) for rid in assigned if rid != seed]
    related.sort(key=lambda x: x[0])
    removed = [seed] + [rid for _, rid in related[: max(0, q - 1)]]
    new_state = clone_state(state)
    for rid in removed:
        new_state = remove_request_from_state(new_state, data, rid)
    return new_state, removed


def regret_repair(state, data, removed, inactive_groups=None, deadline=None):
    new_state = clone_state(state)
    pending = set(removed) | set(new_state.get("unassigned_requests", set()))
    while pending and not time_exceeded(deadline):
        cand_vehicles = candidate_vehicles_for_construction(new_state, data)
        infos = compute_regret_insertions(new_state, data, list(pending), inactive_groups=inactive_groups, vehicle_candidates=cand_vehicles, deadline=deadline)
        feasible_infos = [x for x in infos if x[3] is not None]
        if not feasible_infos:
            return None
        feasible_infos.sort(key=lambda x: (-x[0], x[1]))
        _, _, rid, best = feasible_infos[0]
        _, _, vehicle, route = best
        new_state = assign_request_to_vehicle(new_state, rid, vehicle, route)
        pending.remove(rid)
    return None if pending else new_state


def final_distance_polish(state, data, inactive_groups=None, deadline=None):
    current = clone_state(state)
    current_eval = evaluate_state(data, current, inactive_groups=inactive_groups)
    if not current_eval["feasible"]:
        return current, current_eval
    improved = True
    while improved and not time_exceeded(deadline):
        improved = False
        candidate, cand_eval = vnd(current, data, inactive_groups=inactive_groups, max_rounds=FINAL_VND_ROUNDS, deadline=deadline)
        if cand_eval["feasible"] and cand_eval["search_score"] + 1e-9 < current_eval["search_score"]:
            current, current_eval, improved = candidate, cand_eval, True
            continue
        candidate, cand_eval, changed = tour_depletion_pass(current, data, inactive_groups=inactive_groups, deadline=deadline)
        if changed and cand_eval["feasible"] and cand_eval["search_score"] + 1e-9 < current_eval["search_score"]:
            current, current_eval, improved = candidate, cand_eval, True
    return current, current_eval


def build_and_solve(data, inactive_groups=None, model_name="SAM_SPIDER_REAL_SD_HF_COMPAT_REPEATABLE", output_flag=1):
    rng = random.Random(RANDOM_SEED)
    inactive_groups = set(inactive_groups or [])
    solve_start_time = time.time()
    total_deadline = None if TOTAL_TIME_LIMIT_SEC is None else solve_start_time + TOTAL_TIME_LIMIT_SEC
    search_deadline = main_search_deadline(total_deadline)

    print("SAM/SPIDER: baslangic cozum kuruluyor...")
    current_state, current_eval = build_initial_solution(data, inactive_groups=inactive_groups, deadline=search_deadline)
    if current_state.get("unassigned_requests") and not time_exceeded(search_deadline):
        repaired = regret_repair(current_state, data, removed=[], inactive_groups=inactive_groups, deadline=search_deadline)
        if repaired is not None:
            current_state = repaired
            current_eval = evaluate_state(data, current_state, inactive_groups=inactive_groups)

    best_state = clone_state(current_state)
    best_eval = current_eval
    best_feasible_state, best_feasible_eval = None, None

    if current_eval["feasible"] and not time_exceeded(search_deadline):
        current_state, current_eval, _ = tour_depletion_pass(current_state, data, inactive_groups=inactive_groups, deadline=search_deadline)
        current_state, current_eval = vnd(current_state, data, inactive_groups=inactive_groups, max_rounds=6, deadline=search_deadline)
        best_state, best_eval = clone_state(current_state), current_eval
        best_feasible_state, best_feasible_eval = clone_state(current_state), current_eval

    temperature = SA_START_TEMP
    iteration_count = 0
    print("SAM/SPIDER: ana destroy-repair aramasi basladi...")
    while not time_exceeded(search_deadline):
        roll = rng.random()
        if roll < 0.30:
            destroyed, removed = random_destroy(current_state, data, rng)
        elif roll < 0.55:
            destroyed, removed = shaw_destroy(current_state, data, rng)
        else:
            destroyed, removed = worst_destroy(current_state, data, rng, inactive_groups=inactive_groups, deadline=search_deadline)

        candidate = regret_repair(destroyed, data, removed, inactive_groups=inactive_groups, deadline=search_deadline)
        if candidate is None:
            temperature *= SA_COOLING
            iteration_count += 1
            continue

        cand_eval = evaluate_state(data, candidate, inactive_groups=inactive_groups)
        if cand_eval["feasible"] and not time_exceeded(search_deadline):
            candidate, cand_eval = vnd(candidate, data, inactive_groups=inactive_groups, max_rounds=4, deadline=search_deadline)

        delta = cand_eval["search_score"] - current_eval["search_score"]
        accept = delta <= 0 or rng.random() < math.exp(-delta / max(temperature, 1e-9))
        if accept:
            current_state, current_eval = candidate, cand_eval
            if current_eval["search_score"] + 1e-9 < best_eval["search_score"]:
                best_state, best_eval = clone_state(current_state), current_eval
        if cand_eval["feasible"] and (best_feasible_eval is None or cand_eval["search_score"] + 1e-9 < best_feasible_eval["search_score"]):
            best_feasible_state, best_feasible_eval = clone_state(candidate), cand_eval

        temperature *= SA_COOLING
        iteration_count += 1

    stop_reason = "TOTAL_TIME_LIMIT" if TOTAL_TIME_LIMIT_SEC is not None else "SEARCH_COMPLETE"

    if best_feasible_state is not None and not time_exceeded(total_deadline):
        print("SAM/SPIDER: final polish / intensification basladi...")
        best_feasible_state, best_feasible_eval = final_distance_polish(best_feasible_state, data, inactive_groups=inactive_groups, deadline=total_deadline)

    diagnostics = None
    if best_feasible_state is not None:
        final_state = best_feasible_state
        final_eval = evaluate_state(data, final_state, inactive_groups=inactive_groups, scenario_name=model_name)
    else:
        final_state = best_state
        final_eval = evaluate_state(data, final_state, inactive_groups=inactive_groups, scenario_name=model_name)
        if not final_eval["feasible"]:
            diagnostics = diagnose_infeasibility(
                data,
                final_state,
                final_eval,
                inactive_groups=inactive_groups,
                model_name=model_name,
                stop_reason=stop_reason,
                iteration_count=iteration_count,
            )
            missing = sorted(list(final_state.get("unassigned_requests", set())))
            print("UYARI: Sezgisel arama sonunda feasible cozum bulunamadi.")
            print(f"   Unassigned request sayisi: {len(missing)}")
            print(f"   Ilk 10 unassigned request: {missing[:10]}")
            if diagnostics["unassigned_df"] is not None and len(diagnostics["unassigned_df"]):
                print("   Teshis ozeti (ilk 5 request):")
                for _, r in diagnostics["unassigned_df"].head(5).iterrows():
                    print(f"    - {r['Request_ID']} | {r['Customer']} | Ana neden: {r['Main_Cause']} | ReasonCounts: {r['Aggregated_Reason_Counts']}")

    total_runtime_sec = time.time() - solve_start_time
    if TOTAL_TIME_LIMIT_SEC is not None and total_runtime_sec > TOTAL_TIME_LIMIT_SEC:
        total_runtime_sec = TOTAL_TIME_LIMIT_SEC
    return {
        "state": final_state,
        "eval": final_eval,
        "Status": "FEASIBLE" if final_eval["feasible"] else "INFEASIBLE",
        "ObjVal": final_eval["objective"],
        "SearchScore": final_eval["search_score"],
        "SolCount": 1 if final_eval["feasible"] else 0,
        "Scenario": model_name,
        "StopReason": stop_reason,
        "IterationsDone": iteration_count,
        "RuntimeSec": total_runtime_sec,
        "InfeasibilityDiagnostics": diagnostics,
    }


# =========================
# OUTPUT TABLES + MAP
# =========================
def create_solution_tables(data, result, runtimes=None):
    state = result["state"]
    ev = result["eval"]
    route_rows, route_detail_rows, service_rows, customer_rows, operation_rows = [], [], [], [], []
    routes_for_map = {}

    for v in data["vehicles"]:
        rm = ev["route_metrics"][v]
        route = rm["route"]
        if not rm["used"] or not rm.get("feasible_route", False):
            continue
        physical = rm["physical_vehicle"]
        tour_no = rm["tour_no"]
        loc_route = [route_location(n, data) for n in route]
        map_key = f"{physical} / TUR {tour_no}"
        routes_for_map[map_key] = loc_route
        total_distance = rm["distance"]
        total_km = total_distance * DISTANCE_TO_KM_FACTOR
        variable_cost = rm.get("variable_cost", total_km * data["km_cost"].get(v, 0.0))
        fixed_cost = rm.get("fixed_cost", 0.0)
        total_route_cost = rm.get("total_cost", variable_cost + fixed_cost)

        route_rows.append({
            "Rota_ID": map_key,
            "Plaka": physical,
            "Tur_No": tour_no,
            "Arac_Cinsi": data["vehicle_type"][v],
            "Route": " -> ".join(loc_route),
            "Request_Route": " -> ".join(route),
            "Toplam_Mesafe": safe_round(total_distance, 4),
            "Toplam_Km": safe_round(total_km, 4),
            "Kapasite": data["capacity"][v],
            "Baslangic_Delivery_Yuku": safe_round(rm.get("initial_load", 0.0), 4),
            "Tur_Baslangic": minutes_to_day_hhmm(rm.get("start_time", 0.0)),
            "Tur_Bitis": minutes_to_day_hhmm(rm.get("return_time", 0.0)),
            "Km_Maliyeti": safe_round(data["km_cost"].get(v, 0.0), 4),
            "Sabit_Maliyet": safe_round(fixed_cost, 4),
            "Degisken_Yol_Maliyeti": safe_round(variable_cost, 4),
            "Toplam_Rota_Maliyeti": safe_round(total_route_cost, 4),
            "Optimizasyon_Metrigi": SECONDARY_OBJECTIVE,
        })

        cumulative_distance = 0.0
        prev_loc = data["depot"]
        for seq, stop in enumerate(rm["detail"], start=1):
            node = stop["node"]
            loc = stop["customer"]
            leg_distance = data["distance"][(prev_loc, loc)]
            cumulative_distance += leg_distance
            if node == data["depot"]:
                op_type = "RETURN_TO_DEPOT"
                delivery_text = pickup_text = ""
            else:
                req = data["requests"][node]
                op_type = operation_type(req["delivery_volume"], req["pickup_volume"])
                delivery_text = line_detail_to_text(data["delivery_lines_by_customer"].get(loc, []))
                pickup_text = line_detail_to_text(data["pickup_lines_by_customer"].get(loc, []))
                service_rows.append({
                    "Rota_ID": map_key,
                    "Plaka": physical,
                    "Tur_No": tour_no,
                    "Request_ID": node,
                    "Musteri": loc,
                    "Split_Index": req["split_index"],
                    "Split_Count": req["split_count"],
                    "Operasyon_Tipi": op_type,
                    "Teslim_Hacim": safe_round(req["delivery_volume"], 4),
                    "Pickup_Hacim": safe_round(req["pickup_volume"], 4),
                    "Servis_Baslangic_Dakika": safe_round(stop["service_start"], 2),
                    "Servis_Baslangic_Saat": minutes_to_day_hhmm(stop["service_start"]),
                    "Yuk_Servis_Sonrasi_Hacim": safe_round(stop["load_after"], 4),
                    "Ertesi_Gun_Servis": int(stop.get("next_day_service", 0) or 0),
                    "Teslim_Urun_Detayi": delivery_text,
                    "Pickup_Urun_Detayi": pickup_text,
                })
            route_detail_rows.append({
                "Rota_ID": map_key,
                "Plaka": physical,
                "Tur_No": tour_no,
                "Sira_No": seq,
                "Onceki_Nokta": prev_loc,
                "Nokta": loc,
                "Request_ID": node,
                "Nokta_Tipi": "DEPO" if node == data["depot"] else "MUSTERI",
                "Operasyon_Tipi": op_type,
                "Teslim_Hacim": safe_round(stop["delivery_volume"], 4),
                "Pickup_Hacim": safe_round(stop["pickup_volume"], 4),
                "Kalkis_Dakika": safe_round(stop.get("departure_time", 0.0), 2),
                "Kalkis_Saat": minutes_to_day_hhmm(stop.get("departure_time", 0.0)),
                "Seyahat_Suresi_Dakika": safe_round(stop["travel"], 2),
                "Varis_Dakika": safe_round(stop["arrival"], 2),
                "Varis_Saat": minutes_to_day_hhmm(stop["arrival"]),
                "Servis_Baslangic_Dakika": safe_round(stop["service_start"], 2),
                "Servis_Baslangic_Saat": minutes_to_day_hhmm(stop["service_start"]),
                "Servis_Bitis_Dakika": safe_round(stop["service_end"], 2),
                "Servis_Bitis_Saat": minutes_to_day_hhmm(stop["service_end"]),
                "Bekleme_Dakika": safe_round(stop.get("wait_before_service", 0.0), 2),
                "Ertesi_Gun_Servis": int(stop.get("next_day_service", 0) or 0),
                "Ilk_Gun_Kapanisa_Gore_Gecikme_Dk": safe_round(stop.get("delay_from_day0_close", 0.0), 2),
                "Yuk_Servis_Sonrasi_Hacim": safe_round(stop["load_after"], 4),
                "Mesafe_Bu_Bacak": safe_round(leg_distance, 4),
                "Kumulatif_Mesafe": safe_round(cumulative_distance, 4),
            })
            operation_rows.append({
                "Rota_ID": map_key,
                "Plaka": physical,
                "Tur_No": tour_no,
                "Sira_No": seq,
                "Onceki_Nokta": prev_loc,
                "Nokta": loc,
                "Request_ID": node,
                "Operasyon_Tipi": op_type,
                "Mesafe_Bu_Bacak": safe_round(leg_distance, 4),
                "Kumulatif_Mesafe": safe_round(cumulative_distance, 4),
                "Kalkis_Saat": minutes_to_day_hhmm(stop.get("departure_time", 0.0)),
                "Varis_Saat": minutes_to_day_hhmm(stop["arrival"]),
                "Servis_Baslangic_Saat": minutes_to_day_hhmm(stop["service_start"]),
                "Servis_Bitis_Saat": minutes_to_day_hhmm(stop["service_end"]),
                "Bekleme_Dakika": safe_round(stop.get("wait_before_service", 0.0), 2),
                "Ertesi_Gun_Servis": int(stop.get("next_day_service", 0) or 0),
                "Teslim_Urun_Detayi": delivery_text,
                "Pickup_Urun_Detayi": pickup_text,
                "Noktadan_Ayrilis_Hacmi": safe_round(stop["load_after"], 4),
            })
            prev_loc = loc

    for cust in data["customers"]:
        assigned = [row for row in service_rows if row["Musteri"] == cust]
        customer_rows.append({
            "Musteri": cust,
            "Toplam_Teslim_Hacim": safe_round(sum(row["Teslim_Hacim"] for row in assigned), 4),
            "Toplam_Pickup_Hacim": safe_round(sum(row["Pickup_Hacim"] for row in assigned), 4),
            "Servis_Veren_Rota_Sayisi": len(set(row["Rota_ID"] for row in assigned)),
            "Servis_Veren_Rotalar": ", ".join(sorted(set(row["Rota_ID"] for row in assigned))),
            "Split_Request_Sayisi": len(assigned),
            "Bolge": data["region"].get(cust, ""),
            "Ertesi_Gun_Servis_Adedi": sum(int(row.get("Ertesi_Gun_Servis", 0) or 0) for row in assigned),
        })

    summary_items = [
        {"Bilesen": "Used_Tours", "Deger": ev["used_vehicles"]},
        {"Bilesen": "Used_Physical_Vehicles", "Deger": ev["used_physical_vehicles"]},
        {"Bilesen": "Repeat_Tour_Count", "Deger": ev["repeat_tour_count"]},
        {"Bilesen": "Next_Day_Service_Count", "Deger": ev["next_day_service_count"]},
        {"Bilesen": "Next_Day_Delay_Min", "Deger": safe_round(ev["next_day_delay_min"], 4)},
        {"Bilesen": "Total_Distance", "Deger": safe_round(ev["total_distance"], 4)},
        {"Bilesen": "Total_Fixed_Cost", "Deger": safe_round(ev.get("total_fixed_cost", 0.0), 4)},
        {"Bilesen": "Total_Variable_Cost", "Deger": safe_round(ev.get("total_variable_cost", 0.0), 4)},
        {"Bilesen": "Total_Transport_Cost", "Deger": safe_round(ev.get("total_transport_cost", 0.0), 4)},
        {"Bilesen": "Objective_Type", "Deger": ev.get("objective_type", SECONDARY_OBJECTIVE)},
        {"Bilesen": "Objective_Value", "Deger": safe_round(ev["objective"], 4)},
        {"Bilesen": "Search_Score", "Deger": safe_round(ev["search_score"], 4)},
        {"Bilesen": "Unassigned_Count", "Deger": ev["unassigned_count"]},
        {"Bilesen": "Total_Violation", "Deger": safe_round(ev["total_violation"], 4)},
        {"Bilesen": "StopReason", "Deger": result.get("StopReason", "")},
        {"Bilesen": "IterationsDone", "Deger": result.get("IterationsDone", "")},
        {"Bilesen": "SolverRuntimeSec", "Deger": safe_round(result.get("RuntimeSec", 0.0), 4)},
    ]
    if runtimes:
        for k, v in runtimes.items():
            summary_items.append({"Bilesen": k, "Deger": safe_round(v, 4)})

    objective_df = pd.DataFrame(summary_items)

    return {
        "summary_df": objective_df,
        "route_df": pd.DataFrame(route_rows),
        "route_detail_df": pd.DataFrame(route_detail_rows),
        "service_df": pd.DataFrame(service_rows),
        "customer_df": pd.DataFrame(customer_rows),
        "operation_df": pd.DataFrame(operation_rows),
        "feas_sum_df": ev["feasibility"]["summary"],
        "feas_det_df": ev["feasibility"]["detail"],
        "routes": routes_for_map,
    }


def build_osrm_route_geometry(origin_lonlat, dest_lonlat):
    try:
        url = f"{OSRM_BASE}/route/v1/{OSRM_PROFILE}/{origin_lonlat[0]},{origin_lonlat[1]};{dest_lonlat[0]},{dest_lonlat[1]}"
        params = {"overview": "full", "geometries": "geojson", "steps": "false"}
        resp = requests.get(url, params=params, timeout=OSRM_TIMEOUT)
        resp.raise_for_status()
        data = resp.json()
        if data.get("code") == "Ok" and data.get("routes"):
            coords = data["routes"][0]["geometry"]["coordinates"]
            return [(lat, lon) for lon, lat in coords]
    except Exception:
        pass
    return [(origin_lonlat[1], origin_lonlat[0]), (dest_lonlat[1], dest_lonlat[0])]


def get_vehicle_color(idx):
    palette = ["red", "blue", "green", "purple", "orange", "darkred", "cadetblue", "darkblue", "darkgreen", "pink", "gray", "black", "lightred", "lightblue", "lightgreen", "beige"]
    return palette[idx % len(palette)]


def _map_html_escape(value):
    if value is None:
        return ""
    if isinstance(value, float) and math.isnan(value):
        return ""
    return html.escape(str(value), quote=True)


def _map_fmt(value, nd=2):
    if value is None or value == "":
        return ""
    try:
        if isinstance(value, str):
            txt = value.strip()
            if txt == "":
                return ""
            value = float(txt.replace(",", "."))
        return f"{float(value):,.{nd}f}"
    except Exception:
        return str(value)


def _html_table_from_rows(rows, columns, header_map=None, max_rows=None):
    header_map = header_map or {}
    if not rows:
        return "<i>Kayıt yok.</i>"
    if max_rows is not None:
        rows = rows[:max_rows]
    ths = "".join(
        f"<th style='border:1px solid #ddd;padding:4px;background:#f3f3f3;text-align:left;'>{_map_html_escape(header_map.get(c, c))}</th>"
        for c in columns
    )
    trs = []
    for row in rows:
        tds = "".join(
            f"<td style='border:1px solid #ddd;padding:4px;vertical-align:top;'>{_map_html_escape(row.get(c, ''))}</td>"
            for c in columns
        )
        trs.append(f"<tr>{tds}</tr>")
    return (
        "<table style='border-collapse:collapse;width:100%;font-size:11px;'>"
        f"<thead><tr>{ths}</tr></thead><tbody>{''.join(trs)}</tbody></table>"
    )


def build_route_popup_html(route_id, route_meta, route_detail_df, operation_df, segment_origin=None, segment_dest=None, segment_seq=None):
    route_meta = route_meta or {}
    km_cost = route_meta.get("Km_Maliyeti", 0.0)
    try:
        km_cost = float(km_cost)
    except Exception:
        km_cost = 0.0

    detail_rows = []
    if route_detail_df is not None and not route_detail_df.empty and "Rota_ID" in route_detail_df.columns:
        tmp = route_detail_df[route_detail_df["Rota_ID"] == route_id].copy()
        if "Sira_No" in tmp.columns:
            tmp = tmp.sort_values("Sira_No")
        for _, r in tmp.iterrows():
            leg_distance = r.get("Mesafe_Bu_Bacak", 0.0)
            try:
                leg_km = float(leg_distance) * DISTANCE_TO_KM_FACTOR
            except Exception:
                leg_km = 0.0
            detail_rows.append({
                "Sıra": r.get("Sira_No", ""),
                "Önceki": r.get("Onceki_Nokta", ""),
                "Nokta": r.get("Nokta", ""),
                "Operasyon": r.get("Operasyon_Tipi", ""),
                "Kalkış": r.get("Kalkis_Saat", ""),
                "Süre": _map_fmt(r.get("Seyahat_Suresi_Dakika", ""), 1),
                "Varış": r.get("Varis_Saat", ""),
                "Servis": r.get("Servis_Baslangic_Saat", ""),
                "Bekleme": _map_fmt(r.get("Bekleme_Dakika", ""), 1),
                "Ertesi Gün": r.get("Ertesi_Gun_Servis", ""),
                "Teslim": _map_fmt(r.get("Teslim_Hacim", ""), 2),
                "Pickup": _map_fmt(r.get("Pickup_Hacim", ""), 2),
                "Yük": _map_fmt(r.get("Yuk_Servis_Sonrasi_Hacim", ""), 2),
                "Bacak km": _map_fmt(leg_km, 2),
                "Bacak maliyet": _map_fmt(leg_km * km_cost, 2),
            })

    op_rows = []
    if operation_df is not None and not operation_df.empty and "Rota_ID" in operation_df.columns:
        tmp = operation_df[operation_df["Rota_ID"] == route_id].copy()
        if "Sira_No" in tmp.columns:
            tmp = tmp.sort_values("Sira_No")
        for _, r in tmp.iterrows():
            op_rows.append({
                "Sıra": r.get("Sira_No", ""),
                "Müşteri": r.get("Nokta", ""),
                "Operasyon": r.get("Operasyon_Tipi", ""),
                "Teslim Ürün Detayı": r.get("Teslim_Urun_Detayi", ""),
                "Pickup Ürün Detayı": r.get("Pickup_Urun_Detayi", ""),
            })

    segment_text = ""
    if segment_origin is not None and segment_dest is not None:
        segment_text = f"<div style='margin:4px 0 8px 0;'><b>Tıklanan bacak:</b> {_map_html_escape(segment_seq)} | {_map_html_escape(segment_origin)} → {_map_html_escape(segment_dest)}</div>"

    summary_items = [
        ("Rota", route_id),
        ("Plaka", route_meta.get("Plaka", "")),
        ("Tur No", route_meta.get("Tur_No", "")),
        ("Araç Cinsi", route_meta.get("Arac_Cinsi", "")),
        ("Kapasite", _map_fmt(route_meta.get("Kapasite", ""), 2)),
        ("Başlangıç Delivery Yükü", _map_fmt(route_meta.get("Baslangic_Delivery_Yuku", ""), 2)),
        ("Tur Başlangıç", route_meta.get("Tur_Baslangic", "")),
        ("Tur Bitiş", route_meta.get("Tur_Bitis", "")),
        ("Toplam km", _map_fmt(route_meta.get("Toplam_Km", ""), 2)),
        ("Sabit Maliyet", _map_fmt(route_meta.get("Sabit_Maliyet", ""), 2)),
        ("Km Maliyeti", _map_fmt(route_meta.get("Km_Maliyeti", ""), 2)),
        ("Değişken Yol Maliyeti", _map_fmt(route_meta.get("Degisken_Yol_Maliyeti", ""), 2)),
        ("Toplam Rota Maliyeti", _map_fmt(route_meta.get("Toplam_Rota_Maliyeti", ""), 2)),
    ]
    summary_html = "".join(
        "<tr>"
        f"<td style='border:1px solid #ddd;padding:4px;background:#fafafa;font-weight:bold;'>{_map_html_escape(k)}</td>"
        f"<td style='border:1px solid #ddd;padding:4px;'>{_map_html_escape(v)}</td>"
        "</tr>"
        for k, v in summary_items
    )

    detail_table = _html_table_from_rows(
        detail_rows,
        ["Sıra", "Önceki", "Nokta", "Operasyon", "Kalkış", "Süre", "Varış", "Servis", "Bekleme", "Ertesi Gün", "Teslim", "Pickup", "Yük", "Bacak km", "Bacak maliyet"],
    )
    op_table = _html_table_from_rows(
        op_rows,
        ["Sıra", "Müşteri", "Operasyon", "Teslim Ürün Detayı", "Pickup Ürün Detayı"],
    )

    return f"""
    <div style="width:720px;max-height:520px;overflow-y:auto;font-family:Arial, sans-serif;font-size:12px;">
      <h3 style="margin:0 0 6px 0;">SAM/SPIDER Rota Sevk Detayı</h3>
      {segment_text}
      <table style="border-collapse:collapse;width:100%;font-size:12px;margin-bottom:8px;">
        <tbody>{summary_html}</tbody>
      </table>
      <div style="margin:6px 0;"><b>Rota:</b> {_map_html_escape(route_meta.get("Route", ""))}</div>
      <h4 style="margin:10px 0 4px 0;">Sevk Planı ve Maliyet Kırılımı</h4>
      {detail_table}
      <h4 style="margin:10px 0 4px 0;">Ürün / Operasyon Detayları</h4>
      {op_table}
    </div>
    """


def generate_route_map(data, tables, output_html):
    if folium is None:
        print("folium kurulu degil; harita uretilmedi.")
        return None
    routes = tables.get("routes", {})
    coord_map = data.get("coord_map", {})
    if not routes or not coord_map:
        return None
    center_lat = sum(v["lat"] for v in coord_map.values()) / len(coord_map)
    center_lon = sum(v["lon"] for v in coord_map.values()) / len(coord_map)
    fmap = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles=MAP_TILES, control_scale=True)

    depot = data["depot"]
    dep = coord_map[depot]
    folium.Marker(
        [dep["lat"], dep["lon"]],
        tooltip=f"Depo: {depot}",
        popup=folium.Popup(f"<b>Depo:</b> {_map_html_escape(depot)}", max_width=300),
        icon=folium.Icon(color="black", icon="home"),
    ).add_to(fmap)

    route_df = tables.get("route_df", pd.DataFrame())
    route_detail_df = tables.get("route_detail_df", pd.DataFrame())
    operation_df = tables.get("operation_df", pd.DataFrame())

    for idx, (route_id, route) in enumerate(routes.items()):
        color = get_vehicle_color(idx)
        route_meta = {}
        if not route_df.empty and "Rota_ID" in route_df.columns:
            tmp = route_df[route_df["Rota_ID"] == route_id]
            if not tmp.empty:
                route_meta = tmp.iloc[0].to_dict()

        layer_name = route_id
        if route_meta:
            layer_name = (
                f"{route_id} | {route_meta.get('Arac_Cinsi', '')} | "
                f"{_map_fmt(route_meta.get('Toplam_Km', ''), 1)} km | "
                f"{_map_fmt(route_meta.get('Toplam_Rota_Maliyeti', ''), 0)} maliyet"
            )
        layer = folium.FeatureGroup(name=layer_name, show=True)
        full_route_popup_html = build_route_popup_html(route_id, route_meta, route_detail_df, operation_df)

        for seq in range(len(route) - 1):
            origin, dest = route[seq], route[seq + 1]
            if origin not in coord_map or dest not in coord_map:
                continue
            o, d = coord_map[origin], coord_map[dest]
            points = build_osrm_route_geometry((o["lon"], o["lat"]), (d["lon"], d["lat"]))
            segment_popup_html = build_route_popup_html(route_id, route_meta, route_detail_df, operation_df, segment_origin=origin, segment_dest=dest, segment_seq=f"{seq}->{seq + 1}")
            tooltip_text = f"{route_id}: {origin} → {dest} | Toplam km: {_map_fmt(route_meta.get('Toplam_Km', ''), 1)} | Maliyet: {_map_fmt(route_meta.get('Toplam_Rota_Maliyeti', ''), 0)}"
            folium.PolyLine(points, color=color, weight=6, opacity=0.85, tooltip=tooltip_text, popup=folium.Popup(segment_popup_html, max_width=760)).add_to(layer)

        folium.CircleMarker(
            [dep["lat"], dep["lon"]],
            radius=9,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.7,
            tooltip=f"{route_id} rota özeti",
            popup=folium.Popup(full_route_popup_html, max_width=760),
        ).add_to(layer)

        for seq, node in enumerate(route):
            if node not in coord_map:
                continue
            info = coord_map[node]
            popup_lines = [
                f"<b>{_map_html_escape(route_id)}</b>",
                f"Nokta: {_map_html_escape(node)}",
                f"Sıra: {_map_html_escape(seq)}",
                f"Toplam km: {_map_html_escape(_map_fmt(route_meta.get('Toplam_Km', ''), 2))}",
                f"Rota maliyeti: {_map_html_escape(_map_fmt(route_meta.get('Toplam_Rota_Maliyeti', ''), 2))}",
            ]
            if node != depot and not route_detail_df.empty:
                tmpd = route_detail_df[(route_detail_df["Rota_ID"] == route_id) & (route_detail_df["Nokta"] == node) & (route_detail_df["Sira_No"] == seq)]
                if not tmpd.empty:
                    row = tmpd.iloc[0].to_dict()
                    popup_lines.extend([
                        f"Operasyon: {_map_html_escape(row.get('Operasyon_Tipi',''))}",
                        f"Teslim: {_map_html_escape(_map_fmt(row.get('Teslim_Hacim',''), 2))}",
                        f"Pickup: {_map_html_escape(_map_fmt(row.get('Pickup_Hacim',''), 2))}",
                        f"Servis: {_map_html_escape(row.get('Servis_Baslangic_Saat',''))}",
                        f"Ertesi gün servis: {_map_html_escape(row.get('Ertesi_Gun_Servis',''))}",
                    ])
            folium.CircleMarker(
                [info["lat"], info["lon"]],
                radius=7 if node == depot else 6,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.9,
                tooltip=f"{route_id} | {node} | #{seq}",
                popup=folium.Popup("<br>".join(map(str, popup_lines)), max_width=420),
            ).add_to(layer)
            folium.map.Marker(
                [info["lat"], info["lon"]],
                icon=folium.DivIcon(html=f'<div style="font-size:10pt;color:{color};font-weight:bold;">{seq}-{_map_html_escape(node)}</div>'),
            ).add_to(layer)
        layer.add_to(fmap)

    folium.LayerControl(collapsed=False).add_to(fmap)
    fmap.get_root().html.add_child(folium.Element('<h3 align="center"><b>SAM/SPIDER Gercek Veri Rota Haritasi</b></h3>'))
    fmap.get_root().html.add_child(folium.Element('<p align="center">Sol üstteki katman menüsünden rotaları tek tek açıp kapatabilirsiniz. Rota çizgisine tıklayınca tüm sevk detayı ve maliyet popup içinde gösterilir.</p>'))
    fmap.get_root().html.add_child(folium.Element("""
    <div id="route-layer-toggle-buttons" style="position: fixed; top: 10px; left: 58px; z-index: 999999; background: white; border: 1px solid #bbb; border-radius: 4px; padding: 6px 8px; box-shadow: 0 1px 4px rgba(0,0,0,0.25); font-family: Arial, sans-serif; font-size: 12px;">
      <button type="button" onclick="selectAllRouteLayers(true)" style="margin-right:6px; padding:4px 8px; cursor:pointer;">Tümünü seç</button>
      <button type="button" onclick="selectAllRouteLayers(false)" style="padding:4px 8px; cursor:pointer;">Tümünü kaldır</button>
    </div>
    <script>
    function getRouteLayerCheckboxes() {
      return Array.from(document.querySelectorAll('.leaflet-control-layers-overlays input.leaflet-control-layers-selector'));
    }
    function selectAllRouteLayers(targetState) {
      getRouteLayerCheckboxes().forEach(function(cb) {
        if (cb.checked !== targetState) {
          cb.click();
        }
      });
    }
    (function() {
      var box = document.getElementById('route-layer-toggle-buttons');
      if (box && window.L && L.DomEvent) {
        L.DomEvent.disableClickPropagation(box);
        L.DomEvent.disableScrollPropagation(box);
      }
    })();
    </script>
    """))
    fmap.save(output_html)
    return output_html


def write_solution_output(data, result, runtimes=None):
    tables = create_solution_tables(data, result, runtimes=runtimes)
    with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
        tables["summary_df"].to_excel(writer, sheet_name="summary", index=False)
        tables["route_df"].to_excel(writer, sheet_name="routes", index=False)
        tables["route_detail_df"].to_excel(writer, sheet_name="route_details", index=False)
        tables["service_df"].to_excel(writer, sheet_name="service_assign", index=False)
        tables["customer_df"].to_excel(writer, sheet_name="customer_summary", index=False)
        tables["operation_df"].to_excel(writer, sheet_name="operation_plan", index=False)
        tables["feas_sum_df"].to_excel(writer, sheet_name="feas_summary", index=False)
        tables["feas_det_df"].to_excel(writer, sheet_name="feas_detail", index=False)
        diagnostics = result.get("InfeasibilityDiagnostics")
        if diagnostics:
            if diagnostics.get("summary_df") is not None and len(diagnostics["summary_df"]):
                diagnostics["summary_df"].to_excel(writer, sheet_name="infeas_summary", index=False)
            if diagnostics.get("unassigned_df") is not None and len(diagnostics["unassigned_df"]):
                diagnostics["unassigned_df"].to_excel(writer, sheet_name="infeas_unassigned", index=False)
            if diagnostics.get("vehicle_attempt_df") is not None and len(diagnostics["vehicle_attempt_df"]):
                diagnostics["vehicle_attempt_df"].to_excel(writer, sheet_name="infeas_vehicle_attempt", index=False)
            if diagnostics.get("standalone_df") is not None and len(diagnostics["standalone_df"]):
                diagnostics["standalone_df"].to_excel(writer, sheet_name="infeas_standalone", index=False)
            if diagnostics.get("global_violation_df") is not None and len(diagnostics["global_violation_df"]):
                diagnostics["global_violation_df"].to_excel(writer, sheet_name="infeas_global_viol", index=False)
    print(f"Solution workbook written to: {OUTPUT_FILE}")
    if GENERATE_ROUTE_MAP:
        map_file = generate_route_map(data, tables, ROUTE_MAP_OUTPUT_HTML)
        if map_file:
            print(f"Route map written to: {map_file}")
    return tables


def main():
    total_start = time.time()
    runtimes = {}

    print("1/6 Excel girdileri okunuyor...")
    t0 = time.time()
    raw = read_inputs()
    runtimes["ExcelReadRuntimeSec"] = time.time() - t0
    print(f"   Excel girdileri okundu. Süre: {runtimes['ExcelReadRuntimeSec']:.2f} sn")

    print("2/6 Günlük aktif siparişler ve lokasyon seti hazırlanıyor...")
    t0 = time.time()
    ctx = build_plan_context(raw)
    runtimes["PlanContextRuntimeSec"] = time.time() - t0
    print(f"   Aktif müşteri sayısı: {len(ctx['active_customers'])}")
    print(f"   Aktif lokasyon sayısı (depo dahil): {len(ctx['locations_active'])}")
    print(f"   Süre: {runtimes['PlanContextRuntimeSec']:.2f} sn")

    print("3/6 Google statik mesafe/süre matrisi hazırlanıyor...")
    t0 = time.time()
    dist_df = create_or_load_distance_matrix(ctx["locations_active"], ctx["plan_dt"])
    runtimes["DistanceMatrixRuntimeSec"] = time.time() - t0
    print(f"   Mesafe matrisi hazır. Pair sayısı: {len(dist_df)} | Süre: {runtimes['DistanceMatrixRuntimeSec']:.2f} sn")

    print("4/6 Veri SAM/SPIDER formatına dönüştürülüyor...")
    t0 = time.time()
    data = prepare_data(raw, ctx, dist_df)
    runtimes["PrepareDataRuntimeSec"] = time.time() - t0
    print(f"   Veri hazır. Request sayısı: {len(data['requests'])}, Fiziksel araç: {len(data['physical_vehicles'])}, Sanal tur slotu: {len(data['vehicles'])}")
    print(f"   Süre: {runtimes['PrepareDataRuntimeSec']:.2f} sn")

    print("5/6 SAM/SPIDER sezgisel çözümü başlıyor...")
    t0 = time.time()
    result = build_and_solve(data)
    runtimes["SolverRuntimeSec"] = time.time() - t0
    print(f"   SAM/SPIDER çözümü tamamlandı. Süre: {runtimes['SolverRuntimeSec']:.2f} sn")
    if result.get("Status") == "INFEASIBLE":
        diag = result.get("InfeasibilityDiagnostics") or {}
        unassigned_df = diag.get("unassigned_df", pd.DataFrame())
        print("   UYARI: Cozum INFEASIBLE bitti. Detayli teshis Excel ciktilarina yazilacak.")
        if unassigned_df is not None and len(unassigned_df):
            print("   Ilk 5 teshis satiri:")
            for _, r in unassigned_df.head(5).iterrows():
                print(f"    - {r['Request_ID']} | {r['Customer']} | {r['Main_Cause']}")

    print("6/6 Excel ve harita çıktıları yazılıyor...")
    t0 = time.time()
    write_solution_output(data, result, runtimes=runtimes)
    runtimes["OutputRuntimeSec"] = time.time() - t0
    runtimes["TotalRuntimeSec"] = time.time() - total_start
    print(f"   Çıktılar yazıldı. Süre: {runtimes['OutputRuntimeSec']:.2f} sn")

    print({
        "Status": result["Status"],
        "ObjVal": result["ObjVal"],
        "SearchScore": result["SearchScore"],
        "StopReason": result["StopReason"],
        "IterationsDone": result["IterationsDone"],
        "RuntimeSec": result["RuntimeSec"],
        "ExcelReadRuntimeSec": round(runtimes["ExcelReadRuntimeSec"], 2),
        "PlanContextRuntimeSec": round(runtimes["PlanContextRuntimeSec"], 2),
        "DistanceMatrixRuntimeSec": round(runtimes["DistanceMatrixRuntimeSec"], 2),
        "PrepareDataRuntimeSec": round(runtimes["PrepareDataRuntimeSec"], 2),
        "SolverRuntimeSec": round(runtimes["SolverRuntimeSec"], 2),
        "OutputRuntimeSec": round(runtimes["OutputRuntimeSec"], 2),
        "TotalRuntimeSec": round(runtimes["TotalRuntimeSec"], 2),
        "UnassignedCount": result["eval"].get("unassigned_count", 0),
    })


if __name__ == "__main__":
    main()


1/6 Excel girdileri okunuyor...
   Excel girdileri okundu. Süre: 0.19 sn
2/6 Günlük aktif siparişler ve lokasyon seti hazırlanıyor...
   Aktif müşteri sayısı: 117
   Aktif lokasyon sayısı (depo dahil): 118
   Süre: 0.01 sn
3/6 Google statik mesafe/süre matrisi hazırlanıyor...
Mesafe matrisi cache okundu: /Users/mericanerdogan/Desktop/VRP2/google_distance_matrix_cache.xlsx
   Mesafe matrisi hazır. Pair sayısı: 13924 | Süre: 22.83 sn
4/6 Veri SAM/SPIDER formatına dönüştürülüyor...
   Veri hazır. Request sayısı: 118, Fiziksel araç: 115, Sanal tur slotu: 345
   Süre: 9.41 sn
5/6 SAM/SPIDER sezgisel çözümü başlıyor...
SAM/SPIDER: baslangic cozum kuruluyor...
SAM/SPIDER: ana destroy-repair aramasi basladi...
   SAM/SPIDER çözümü tamamlandı. Süre: 4177.15 sn
6/6 Excel ve harita çıktıları yazılıyor...
Solution workbook written to: /Users/mericanerdogan/Desktop/VRP2/sam_spider_real_data_output_28_04.xlsx
Route map written to: /Users/mericanerdogan/Desktop/VRP2/sam_spider_real_data_route_map_28_